In [ ]:
# ================== 1. Setup & Packages ==================
!apt-get -qq -y install fonts-nanum > /dev/null
!pip install -q tensorflow

import os, time, gc, random, logging, math, subprocess
from datetime import timedelta
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
import matplotlib.font_manager as fm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

from tqdm.notebook import tqdm
from sklearn.metrics import r2_score, mean_squared_error

# ================== Global Flags ==================
SEED = 42
DEBUG_PLOTS   = True   # True로 하면 test set t0 normalized debug plot 저장
VERBOSE_MISS  = False   # True로 하면 env/ir missing 로그 자세히 출력

# ---- Deterministic env settings (must be before TF import) ----
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf  # ✅ TF는 여기서 처음 import
tf.get_logger().setLevel(logging.ERROR)

# ---- Seed reset helper (for reproducibility) ----
def reset_seeds(seed=SEED):
    np.random.seed(seed)
    random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism(True)
    except Exception:
        pass

def configure_tf_runtime(seed=SEED, set_memory_growth=True, set_threads=True):
    # GPU memory growth: Colab OOM 완화
    if set_memory_growth:
        try:
            gpus = tf.config.list_physical_devices("GPU")
            for gpu in gpus:
                try:
                    tf.config.experimental.set_memory_growth(gpu, True)
                except Exception:
                    pass
        except Exception:
            pass

    # Determinism
    try:
        tf.config.experimental.enable_op_determinism(True)
    except Exception:
        pass

    # Threading: 재현성/스파이크 완화(원하면 주석 처리)
    if set_threads:
        try:
            tf.config.threading.set_intra_op_parallelism_threads(1)
            tf.config.threading.set_inter_op_parallelism_threads(1)
        except Exception:
            pass

    tf.keras.backend.set_floatx("float32")
    reset_seeds(seed)

# 세션 정리(HP loop 누수 완화)
def cleanup_tf(model=None):
    try:
        if model is not None:
            del model
    except Exception:
        pass
    try:
        tf.keras.backend.clear_session()
    except Exception:
        pass
    gc.collect()

configure_tf_runtime(SEED)

# ---- Keras imports (✅ TF import 이후에만) ----
from tensorflow.keras import backend as K
from tensorflow.keras.activations import swish
from tensorflow.keras.layers import (
    Layer, Input, Dense, Conv2D, ConvLSTM2D, Reshape, Concatenate,
    Dropout, AveragePooling2D, TimeDistributed, Permute,
    MultiHeadAttention, Add, LayerNormalization, Resizing
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import AdamW

# ---- Fonts (Nanum) ----
font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

def _vprint(msg):
    if VERBOSE_MISS:
        print(msg)

# ================== 2. Data Path ==================
from google.colab import drive
drive.mount("/content/drive")

drive_root    = "/content/drive/MyDrive/Colab"
drive_env_dir = f"{drive_root}/env_data"     # env_data 폴더: SST/TOPO/ERH... + WNP_YYYY.txt 가정

# --- ENV 바이너리(.tar) 평면 추출 로컬 디렉토리 ---
local_env_dir = "/content/env_data/background"
os.makedirs(local_env_dir, exist_ok=True)

# --- 트랙 파일 로컬 복사 경로 ---
TRACK_DIR = "/content/tracks"
os.makedirs(TRACK_DIR, exist_ok=True)

# ✅ 연도별 트랙 파일 복사 (WNP_2016.txt ~ WNP_2024.txt)
for y in range(2016, 2025):
    src = os.path.join(drive_env_dir, f"WNP_{y}.txt")
    dst = os.path.join(TRACK_DIR,    f"WNP_{y}.txt")
    if not os.path.exists(src):
        print(f"[WARN] track not found on drive: {src}")
        continue
    if not os.path.exists(dst):
        print(f"[COPY] {src} -> {dst}")
        subprocess.run(["cp", src, dst], check=False)

# --- 모든 연도 tar를 한 폴더(local_env_dir)에 평면 추출 (이미 풀었으면 건너뜀) ---
years = range(2016, 2025)
for y in years:
    tar_on_drive = f"{drive_env_dir}/{y}.tar"
    if not os.path.exists(tar_on_drive):
        print(f"[!] tar not found: {tar_on_drive}")
        continue

    stamp = os.path.join(local_env_dir, f".extracted_{y}.stamp")
    if os.path.exists(stamp) and os.path.getmtime(stamp) >= os.path.getmtime(tar_on_drive):
        print(f"[SKIP] {y}.tar 이미 추출됨 → {local_env_dir}")
        continue

    print(f"[EXTRACT(flat)] {tar_on_drive} → {local_env_dir}")
    ret = subprocess.run(["tar", "-xf", tar_on_drive, "-C", local_env_dir]).returncode
    if ret == 0:
        with open(stamp, "w") as f:
            f.write("ok\n")
        os.utime(stamp, (time.time(), os.path.getmtime(tar_on_drive)))
    else:
        print(f"[ERR] tar extract failed: {tar_on_drive} (exit={ret})")

# ================== RESOLUTION SWITCH ==================
def get_grid_setup(env_res_deg=0.18, ir_native_deg=0.018, window_deg=18.0):
    env_win_px = int(round(window_deg / float(env_res_deg)))
    if env_win_px % 2 == 0:
        env_win_px += 1
    env_span_deg = env_win_px * float(env_res_deg)
    ir_crop_px   = int(round(env_span_deg / float(ir_native_deg)))
    cx = cy = 600  # IR 1201×1201 중심(가정)
    return {
        "ENV_RES_DEG": float(env_res_deg),
        "ENV_WIN_PX": int(env_win_px),
        "ENV_SPAN_DEG": float(env_span_deg),
        "IR_NATIVE_DEG": float(ir_native_deg),
        "IR_CROP_PX": int(ir_crop_px),
        "CENTER_X": int(cx),
        "CENTER_Y": int(cy),
        "TARGET_H": int(env_win_px),
        "TARGET_W": int(env_win_px),
        "IR_POOL_K": 1,
        "RES_MODE": f"env{env_res_deg}_ir{ir_native_deg}_span{env_span_deg:.3f}",
    }

GRID = get_grid_setup(env_res_deg=0.25, ir_native_deg=0.018, window_deg=18.0)
print("[RESOLUTION]", GRID)

# ================== 3. Variable Settings ==================
multi_level_dict = {
    "ERH":   ["850", "700", "500", "300", "200"],
    "EDIVE": ["850", "700", "500", "300", "200"],
    "EVORT": ["850", "700", "500", "300", "200"],
    "EVWS":  ["850", "700", "500", "300", "200"],
    "ETHETA":  ["850", "700", "500", "300", "200"],
}
columns   = ["basin", "id", "datetime", "latitude", "longitude", "mws", "mslp"]
level_ids = {"850": 0, "700": 1, "500": 2, "300": 3, "200": 4}

# ================== 4. Utilities ==================
def center_crop_or_resize_by_grid(arr2d, src_res_deg, target_hw, span_deg=None):
    """
    arr2d: (H,W) 원본 2D 필드 (이미 태풍 중심 기준 정렬된 정사각이라고 가정)
    src_res_deg: 원본 해상도 (ENV=0.25, IR=0.018)
    target_hw: 최종 출력 크기 (GRID["TARGET_H"])
    span_deg: 잘라낼 실제 degree span. None이면 target_hw * src_res_deg로 계산

    동작:
      1) 중심 기준으로 span_deg에 해당하는 픽셀 크기만큼 crop
      2) crop 결과를 target_hw x target_hw로 resize (필요시)
    """
    if arr2d is None:
        return None
    arr2d = np.asarray(arr2d, dtype=np.float32)
    if arr2d.ndim != 2:
        return None

    H, W = arr2d.shape
    if H <= 0 or W <= 0:
        return None

    if span_deg is None:
        span_deg = float(target_hw) * float(src_res_deg)

    # span_deg에 해당하는 source pixel 수
    crop_px = int(round(float(span_deg) / float(src_res_deg)))
    crop_px = max(1, crop_px)

    # 중심 포함 위해 홀수 선호 (ENV/IR 모두 일관성)
    if crop_px % 2 == 0:
        crop_px += 1

    # 원본보다 크면 그냥 전체 사용 후 resize
    crop_px = min(crop_px, H, W)

    cy = H // 2
    cx = W // 2
    half = crop_px // 2

    y0 = cy - half
    y1 = cy + half + 1
    x0 = cx - half
    x1 = cx + half + 1

    # 경계 보정
    if y0 < 0:
        y1 += (-y0); y0 = 0
    if x0 < 0:
        x1 += (-x0); x0 = 0
    if y1 > H:
        y0 -= (y1 - H); y1 = H
    if x1 > W:
        x0 -= (x1 - W); x1 = W

    y0 = max(0, y0); x0 = max(0, x0)
    cropped = arr2d[y0:y1, x0:x1]

    if cropped.size == 0:
        return None

    # 혹시 shape mismatch면 중앙 강제 재보정
    ch, cw = cropped.shape
    if ch != crop_px or cw != crop_px:
        use = min(ch, cw)
        if use <= 0:
            return None
        if use % 2 == 0:
            use -= 1
        cy2 = ch // 2
        cx2 = cw // 2
        h2 = use // 2
        cropped = cropped[cy2-h2:cy2+h2+1, cx2-h2:cx2+h2+1]

    # 최종 target_hw 맞춤
    if cropped.shape != (target_hw, target_hw):
        interp = cv2.INTER_AREA if min(cropped.shape) >= target_hw else cv2.INTER_CUBIC
        cropped = cv2.resize(cropped, (target_hw, target_hw), interpolation=interp)

    return cropped.astype(np.float32)

def load_filtered_df_years(track_dir, years):
    """
    WNP_YYYY.txt 로더 (RMW 사용 안 함 버전)
    필요한 컬럼:
      basin, id, datetime, latitude, longitude, mws, mslp
    """
    dfs = []

    for y in years:
        path = os.path.join(track_dir, f"WNP_{y}.txt")
        if not os.path.exists(path):
            print(f"[WARN] track file not found: {path}")
            continue

        try:
            raw = pd.read_csv(
                path,
                sep=r"\s+",
                header=None,
                engine="python"
            )
        except Exception as e:
            print(f"[WARN] failed to read track file: {path} | {e}")
            continue

        # 최소 앞 7개 컬럼 필요
        if raw.shape[1] < 7:
            print(f"[WARN] unexpected column count in {path}: {raw.shape[1]}")
            continue

        try:
            df_y = pd.DataFrame({
                "basin": raw.iloc[:, 0],
                "id": raw.iloc[:, 1],
                "datetime": raw.iloc[:, 2],
                "latitude": raw.iloc[:, 3],
                "longitude": raw.iloc[:, 4],
                "mws": raw.iloc[:, 5],
                "mslp": raw.iloc[:, 6],
            })
        except Exception as e:
            print(f"[WARN] failed to slice columns from track file: {path} | {e}")
            continue

        # 타입 정리
        df_y["basin"] = df_y["basin"].astype(str).str.strip()
        for c in ["id", "datetime", "latitude", "longitude", "mws", "mslp"]:
            df_y[c] = pd.to_numeric(df_y[c], errors="coerce")

        # datetime 파싱 (YYYYMMDDHH 10자리)
        dt_num = df_y["datetime"].copy()
        dt_num = dt_num.where(np.isfinite(dt_num), np.nan)
        dt_str = dt_num.map(
            lambda v: f"{int(v):010d}" if (pd.notna(v) and np.isfinite(v) and int(v) > 0) else np.nan
        )
        df_y["datetime"] = pd.to_datetime(dt_str, format="%Y%m%d%H", errors="coerce")

        # 필수값 결측 제거 (RMW 제외)
        before = len(df_y)
        df_y = df_y.dropna(subset=["basin", "id", "datetime", "latitude", "longitude", "mws", "mslp"]).copy()
        dropped = before - len(df_y)
        if dropped > 0:
            print(f"[WARN] dropped invalid rows in {os.path.basename(path)}: {dropped}")

        # 최종 타입
        df_y["id"] = df_y["id"].astype(int)
        for c in ["latitude", "longitude", "mws", "mslp"]:
            df_y[c] = df_y[c].astype(np.float32)

        dfs.append(df_y)

    if len(dfs) == 0:
        print(f"[WARN] No track rows loaded for years={years}")
        return pd.DataFrame(columns=["basin", "id", "datetime", "latitude", "longitude", "mws", "mslp"])

    df = pd.concat(dfs, ignore_index=True)
    df = df.sort_values(["basin", "id", "datetime"]).reset_index(drop=True)
    return df

def load_background(path, crop=True, target_hw=None):
    if not os.path.exists(path):
        return None
    try:
        buf = np.fromfile(path, dtype=np.float32)
        H = int(np.sqrt(buf.size))
        if H * H != buf.size:
            return None

        arr = buf.reshape((H, H)).astype(np.float32)

        if not crop:
            return arr

        if target_hw is None:
            target_hw = int(GRID["TARGET_H"])

        # ✅ GRID 기반 중심 crop + resize (ENV 해상도 사용)
        arr_out = center_crop_or_resize_by_grid(
            arr2d=arr,
            src_res_deg=float(GRID["ENV_RES_DEG"]),
            target_hw=int(target_hw),
            span_deg=float(GRID["ENV_SPAN_DEG"]),
        )
        return arr_out
    except Exception:
        return None

def load_irwvln(path):
    if not os.path.exists(path):
        return None
    try:
        buf = np.fromfile(path, dtype=np.float32)
        L = int(round(np.sqrt(buf.size)))
        if L * L != buf.size:
            return None
        return buf.reshape((L, L))
    except Exception:
        return None

def debug_plot_all_first_normalized_fields_save_and_show_T0(
    X_env_norm, X_ir_norm, save_root, meta_list=None,
    target_id=10,
    max_cases=None,        # None이면 해당 target_id 전체 t0 시점 저장
    sort_by="t0_asc",      # "t0_asc" or "t0_desc"
    show=False,
    fixed_limits=None,     # 유지하되, 이 버전에서는 내부 고정 범위 우선 사용
):
    """
    DEBUG plot 저장 함수

    [CURRENT VERSION]
    - 함수명/호출부 변경 없음
    - target_id 기본값 21
    - max_cases=None이면 해당 target_id 전체 t0 저장
    - IRWVLN, SST, TOPO는 개별 저장
    - ERH/EDIVE/EVORT/EVWS/ETHETA는 기압면별로 각각 저장
    - 범례(colorbar)는 그림 안에 포함하지 않고 별도 PNG로 저장
    - colormap은 전부 viridis

    Fixed display range:
      - SST:   ocean only, land/invalid=black,  -0.5 ~ 0.5, tick=0.1
      - TOPO:  land only,  ocean=black,          0.0 ~ 1.0, tick=0.1
      - IRWVLN:                                  -3.0 ~ 3.0, tick=1.0
      - 3D vars:                                 -1.0 ~ 1.0, tick=0.5

    Colorbar:
      - standalone PNG
      - tick label bold
      - larger physical size
      - high DPI
      - white tick/outline for black-background composite figures

    Notes:
      - SST invalid/land sentinel = -10.0
      - TOPO는 normalize_features 기준으로 ocean이 0.0이라서
        debug plot에서는 img0 > 0 을 land로 간주
    """
    if not DEBUG_PLOTS:
        return

    # ===============================
    # Style settings
    # ===============================
    FIGSIZE_SINGLE = (6.0, 6.0)
    CBAR_FMT = "%.1f"
    SST_SENTINEL_Z = -10.0

    CMAP_MAIN = "viridis"

    # field에서 masking 영역 표시용 black
    BLACK_OVERLAY_CMAP = ListedColormap([(0.0, 0.0, 0.0, 1.0)])

    # standalone colorbar style
    CBAR_TICK_FONTSIZE = 24
    CBAR_TICK_COLOR = "black"
    CBAR_DPI = 600

    out_root = os.path.join(save_root, "debug")
    os.makedirs(out_root, exist_ok=True)

    field_root = os.path.join(out_root, "fields")
    os.makedirs(field_root, exist_ok=True)

    cbar_root = os.path.join(out_root, "colorbars")
    os.makedirs(cbar_root, exist_ok=True)

    base_levels_map = {
        "ERH":    ["850", "700", "500", "300", "200"],
        "EDIVE":  ["850", "700", "500", "300", "200"],
        "EVORT":  ["850", "700", "500", "300", "200"],
        "EVWS":   ["850", "700", "500", "300", "200"],
        "ETHETA": ["850", "700", "500", "300", "200"],
    }

    multi_level_vars = tuple(base_levels_map.keys())

    # ===============================
    # Fixed display spec
    # ===============================
    def _get_fixed_debug_spec(vname):
        vname = str(vname).upper()

        # 3D vars
        if vname in ["ERH", "EDIVE", "EVORT", "EVWS", "ETHETA"]:
            vmin = -1.0
            vmax =  1.0
            ticks = np.arange(-1.0, 1.0 + 0.0001, 0.5, dtype=np.float32)
            return vmin, vmax, ticks, CMAP_MAIN

        # SST
        if vname == "SST":
            vmin = -0.5
            vmax =  0.5
            ticks = np.arange(-0.5, 0.5 + 0.0001, 0.1, dtype=np.float32)
            return vmin, vmax, ticks, CMAP_MAIN

        # TOPO
        if vname == "TOPO":
            vmin = 0.0
            vmax = 1.0
            ticks = np.arange(0.0, 1.0 + 0.0001, 0.1, dtype=np.float32)
            return vmin, vmax, ticks, CMAP_MAIN

        # IRWVLN
        if vname in ["IR", "IRWVLN"]:
            vmin = -3.0
            vmax =  3.0
            ticks = np.arange(-3.0, 3.0 + 0.0001, 1.0, dtype=np.float32)
            return vmin, vmax, ticks, CMAP_MAIN

        return None

    def _fallback_limits(img):
        vals = np.asarray(img)
        vals = vals[np.isfinite(vals)]

        if vals.size == 0:
            return -1.0, 1.0, np.linspace(-1.0, 1.0, 6).astype(np.float32)

        vmin = float(np.nanpercentile(vals, 2))
        vmax = float(np.nanpercentile(vals, 98))

        if (not np.isfinite(vmin)) or (not np.isfinite(vmax)) or (vmin == vmax):
            vmin = float(np.nanmin(vals))
            vmax = float(np.nanmax(vals))

        if (not np.isfinite(vmin)) or (not np.isfinite(vmax)) or (vmin == vmax):
            vmin, vmax = -1.0, 1.0

        ticks = np.linspace(vmin, vmax, 6).astype(np.float32)
        return vmin, vmax, ticks

    def _save_standalone_colorbar(save_path, vmin, vmax, ticks, cmap):
        """
        편집용 standalone colorbar 저장.

        - 기존보다 약 1.5배 큰 물리적 크기
        - tick label bold
        - dpi=600
        - black-background composite에 맞게 white tick/outline
        """
        fig = plt.figure(figsize=(1.9, 6.6), facecolor="none")

        # [left, bottom, width, height]
        ax = fig.add_axes([0.30, 0.06, 0.24, 0.88])

        norm = plt.Normalize(vmin=vmin, vmax=vmax)
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])

        cbar = fig.colorbar(
            sm,
            cax=ax,
            orientation="vertical",
            ticks=ticks,
            format=CBAR_FMT
        )

        cbar.set_ticks(ticks)
        cbar.set_ticklabels([f"{float(t):.1f}" for t in ticks])

        cbar.ax.tick_params(
            labelsize=CBAR_TICK_FONTSIZE,
            colors=CBAR_TICK_COLOR,
            width=1.6,
            length=5,
            pad=5
        )

        # tick label bold
        for lab in cbar.ax.get_yticklabels():
            lab.set_fontweight("bold")
            lab.set_color(CBAR_TICK_COLOR)

        cbar.outline.set_edgecolor(CBAR_TICK_COLOR)
        cbar.outline.set_linewidth(1.4)
        cbar.set_label("")

        plt.savefig(
            save_path,
            dpi=CBAR_DPI,
            bbox_inches="tight",
            transparent=True,
            pad_inches=0.03
        )
        plt.close(fig)

    def _save_field_only(
        save_path,
        img,
        vmin,
        vmax,
        cmap,
        overlay_mask=None,
        overlay_cmap=None,
    ):
        """
        field 그림만 저장.
        colorbar는 포함하지 않음.

        overlay_mask:
          - 값이 1인 영역만 overlay 표시
          - SST의 육지/invalid, TOPO의 해양 등에 사용
        """
        fig, ax = plt.subplots(figsize=FIGSIZE_SINGLE)

        ax.imshow(
            img,
            origin="lower",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )

        if overlay_mask is not None:
            if overlay_cmap is None:
                overlay_cmap = BLACK_OVERLAY_CMAP

            ax.imshow(
                overlay_mask,
                origin="lower",
                cmap=overlay_cmap,
                vmin=0,
                vmax=1
            )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title("")

        plt.tight_layout()
        plt.savefig(save_path, dpi=180, bbox_inches="tight")

        if show:
            plt.show()

        plt.close(fig)

    # ===============================
    # Select target_id cases
    # ===============================
    if meta_list is None or len(meta_list) == 0:
        print(f"[DEBUG] meta_list가 없어 target_id={int(target_id)} 필터 불가. sample0만 저장합니다.")
        selected = [(0, "sample0")]
    else:
        cand = []

        for i, m in enumerate(meta_list):
            try:
                bid = int(m.get("id", -999))
            except Exception:
                continue

            if bid != int(target_id):
                continue

            t0 = m.get("t0", None)
            if t0 is None:
                continue

            cand.append((i, pd.to_datetime(t0)))

        if len(cand) == 0:
            print(f"[DEBUG] id={target_id} 조건 샘플이 없습니다. sample0만 저장합니다.")
            selected = [(0, "sample0")]
        else:
            reverse = True if str(sort_by).lower().endswith("desc") else False
            cand = sorted(cand, key=lambda x: x[1], reverse=reverse)

            if max_cases is not None:
                cand = cand[:int(max_cases)]

            selected = []
            for rank, (idx, t0) in enumerate(cand, start=1):
                basin = str(meta_list[idx].get("basin", ""))
                tag = f"{rank:03d}_{basin}{str(target_id).zfill(2)}_{t0.strftime('%Y%m%d%H')}"
                selected.append((idx, tag))

            case_msg = "all" if max_cases is None else f"top{int(max_cases)}"
            print(
                f"[DEBUG] Selected target_id={int(target_id)} cases={len(selected)} "
                f"({case_msg}) | sort={sort_by} | out={out_root}"
            )

    # ===============================
    # Plot one sample
    # ===============================
    def _plot_for_index(sample_idx, tag):

        # -------------------------------------------------
        # 1. IRWVLN
        # -------------------------------------------------
        if isinstance(X_ir_norm, np.ndarray) and X_ir_norm.shape[0] > sample_idx:
            img = None

            # run 함수에서 Xte_ir[:, 1] 넘기는 경우: (B,H,W,1)
            if X_ir_norm.ndim == 4:
                img = X_ir_norm[sample_idx, :, :, 0]

            # 혹시 전체 pair가 넘어오는 경우: (B,2,H,W,1)
            elif X_ir_norm.ndim == 5:
                tidx = 1 if X_ir_norm.shape[1] > 1 else 0
                img = X_ir_norm[sample_idx, tidx, :, :, 0]

            if img is not None:
                spec = _get_fixed_debug_spec("IRWVLN")
                if spec is not None:
                    vmin, vmax, ticks, cmap_use = spec
                else:
                    vmin, vmax, ticks = _fallback_limits(img)
                    cmap_use = CMAP_MAIN

                img_path = os.path.join(field_root, f"{tag}_IRWVLN_t0_norm.png")
                cbar_path = os.path.join(cbar_root, f"{tag}_IRWVLN_t0_norm_cbar.png")

                _save_field_only(
                    save_path=img_path,
                    img=img,
                    vmin=vmin,
                    vmax=vmax,
                    cmap=cmap_use,
                    overlay_mask=None
                )

                _save_standalone_colorbar(
                    save_path=cbar_path,
                    vmin=vmin,
                    vmax=vmax,
                    ticks=ticks,
                    cmap=cmap_use
                )

        # -------------------------------------------------
        # 2. ENV variables
        # -------------------------------------------------
        for vname, arr in X_env_norm.items():

            if not (
                isinstance(arr, np.ndarray)
                and arr.ndim == 5
                and arr.shape[0] > sample_idx
                and arr.shape[1] > 0
            ):
                continue

            vname_u = str(vname).upper()

            _, Tin, L, H, W = arr.shape

            # ENV 입력은 [-6, 0, +6, ...]
            # t0는 index=1
            t_idx = 1 if Tin > 1 else 0

            # ---------------------------------------------
            # 2-1. SST: ocean only, land=black
            # ---------------------------------------------
            if vname_u == "SST":
                img0 = arr[sample_idx, t_idx, 0]

                valid_ocean_mask = np.isfinite(img0) & (~np.isclose(img0, SST_SENTINEL_Z))
                ocean_img = np.ma.masked_where(~valid_ocean_mask, img0)

                spec = _get_fixed_debug_spec("SST")
                if spec is not None:
                    vmin, vmax, ticks, cmap_use = spec
                else:
                    if np.any(valid_ocean_mask):
                        vmin, vmax, ticks = _fallback_limits(img0[valid_ocean_mask])
                    else:
                        vmin, vmax, ticks = _fallback_limits(img0)
                    cmap_use = CMAP_MAIN

                # 육지/invalid는 black
                overlay_black = None
                if np.any(~valid_ocean_mask):
                    overlay_black = np.where(~valid_ocean_mask, 1.0, np.nan)

                img_path = os.path.join(field_root, f"{tag}_SST_ocean_t0_norm.png")
                cbar_path = os.path.join(cbar_root, f"{tag}_SST_ocean_t0_norm_cbar.png")

                _save_field_only(
                    save_path=img_path,
                    img=ocean_img,
                    vmin=vmin,
                    vmax=vmax,
                    cmap=cmap_use,
                    overlay_mask=overlay_black,
                    overlay_cmap=BLACK_OVERLAY_CMAP,
                )

                _save_standalone_colorbar(
                    save_path=cbar_path,
                    vmin=vmin,
                    vmax=vmax,
                    ticks=ticks,
                    cmap=cmap_use
                )
                continue

            # ---------------------------------------------
            # 2-2. TOPO: land only, ocean=black
            # ---------------------------------------------
            if vname_u == "TOPO":
                img0 = arr[sample_idx, t_idx, 0]

                # normalize_features 상 ocean=0 이므로 land는 >0으로 간주
                land_mask = np.isfinite(img0) & (img0 > 0.0)
                land_img = np.ma.masked_where(~land_mask, img0)

                spec = _get_fixed_debug_spec("TOPO")
                if spec is not None:
                    vmin, vmax, ticks, cmap_use = spec
                else:
                    if np.any(land_mask):
                        vmin, vmax, ticks = _fallback_limits(img0[land_mask])
                    else:
                        vmin, vmax, ticks = 0.0, 1.0, np.arange(0.0, 1.0 + 0.0001, 0.1, dtype=np.float32)
                    cmap_use = CMAP_MAIN

                # 해양은 black
                overlay_black = None
                if np.any(~land_mask):
                    overlay_black = np.where(~land_mask, 1.0, np.nan)

                img_path = os.path.join(field_root, f"{tag}_TOPO_land_t0_norm.png")
                cbar_path = os.path.join(cbar_root, f"{tag}_TOPO_land_t0_norm_cbar.png")

                _save_field_only(
                    save_path=img_path,
                    img=land_img,
                    vmin=vmin,
                    vmax=vmax,
                    cmap=cmap_use,
                    overlay_mask=overlay_black,
                    overlay_cmap=BLACK_OVERLAY_CMAP,
                )

                _save_standalone_colorbar(
                    save_path=cbar_path,
                    vmin=vmin,
                    vmax=vmax,
                    ticks=ticks,
                    cmap=cmap_use
                )
                continue

            # ---------------------------------------------
            # 2-3. 3D multi-level variables
            # ---------------------------------------------
            if vname_u in multi_level_vars:
                base_levels = base_levels_map[vname_u]
                lv_names = (
                    base_levels * ((L + len(base_levels) - 1) // len(base_levels))
                )[:L]

                spec = _get_fixed_debug_spec(vname_u)
                if spec is not None:
                    vmin_fix, vmax_fix, ticks_fix, cmap_use = spec
                else:
                    vmin_fix, vmax_fix, ticks_fix = None, None, None
                    cmap_use = CMAP_MAIN

                for l in range(L):
                    img0 = arr[sample_idx, t_idx, l]

                    if spec is not None:
                        vmin, vmax, ticks = vmin_fix, vmax_fix, ticks_fix
                    else:
                        vmin, vmax, ticks = _fallback_limits(img0)

                    lv = str(lv_names[l]).replace("/", "-")

                    img_path = os.path.join(field_root, f"{tag}_{vname_u}_{lv}hPa_t0_norm.png")
                    cbar_path = os.path.join(cbar_root, f"{tag}_{vname_u}_{lv}hPa_t0_norm_cbar.png")

                    _save_field_only(
                        save_path=img_path,
                        img=img0,
                        vmin=vmin,
                        vmax=vmax,
                        cmap=cmap_use,
                        overlay_mask=None
                    )

                    _save_standalone_colorbar(
                        save_path=cbar_path,
                        vmin=vmin,
                        vmax=vmax,
                        ticks=ticks,
                        cmap=cmap_use
                    )

                continue

            # ---------------------------------------------
            # 2-4. fallback
            # ---------------------------------------------
            lv_names = [f"lv{i}" for i in range(L)]

            spec = _get_fixed_debug_spec(vname_u)
            if spec is not None:
                vmin_fix, vmax_fix, ticks_fix, cmap_use = spec
            else:
                vmin_fix, vmax_fix, ticks_fix = None, None, None
                cmap_use = CMAP_MAIN

            for l in range(L):
                img0 = arr[sample_idx, t_idx, l]

                if spec is not None:
                    vmin, vmax, ticks = vmin_fix, vmax_fix, ticks_fix
                else:
                    vmin, vmax, ticks = _fallback_limits(img0)

                lv = str(lv_names[l]).replace("/", "-")

                img_path = os.path.join(field_root, f"{tag}_{vname_u}_{lv}_t0_norm.png")
                cbar_path = os.path.join(cbar_root, f"{tag}_{vname_u}_{lv}_t0_norm_cbar.png")

                _save_field_only(
                    save_path=img_path,
                    img=img0,
                    vmin=vmin,
                    vmax=vmax,
                    cmap=cmap_use,
                    overlay_mask=None
                )

                _save_standalone_colorbar(
                    save_path=cbar_path,
                    vmin=vmin,
                    vmax=vmax,
                    ticks=ticks,
                    cmap=cmap_use
                )

    # ===============================
    # Execute
    # ===============================
    for idx, tag in selected:
        _plot_for_index(idx, tag)

    case_msg = "all" if max_cases is None else f"top{int(max_cases)}"

    print(f"[DEBUG] Saved field-only plots -> {field_root}")
    print(f"[DEBUG] Saved standalone colorbars -> {cbar_root}")
    print(f"[DEBUG] Saved target_id={int(target_id)} {case_msg} t0 plots.")

def compute_ri_rw_contingency(
    y_true_mws, y_pred_mws, mask, lead_hours,
    ri_thresh=30.0, rw_thresh=30.0, horizon_hours=24,
    min_intensity_kt=34.0,
    ts_on="both",
):
    """
    RI / RW 이진분류 성능지표 계산

    반환되는 dict(ri_stats, rw_stats)에 포함되는 지표:
      - H, M, F, CN
      - POD  : Probability of Detection
      - FAR  : False Alarm Ratio
      - CSI  : Critical Success Index
      - BIAS : Frequency Bias
      - ACC  : Accuracy
      - PRECISION : Precision
      - F1   : F1-score
      - POFD : Probability of False Detection
      - TSS  : True Skill Statistic
      - ETS  : Equitable Threat Score
      - HSS  : Heidke Skill Score
    """
    y_true_mws = np.asarray(y_true_mws, dtype=np.float32)
    y_pred_mws = np.asarray(y_pred_mws, dtype=np.float32)
    mask       = np.asarray(mask, dtype=np.float32)

    assert y_true_mws.shape == y_pred_mws.shape == mask.shape
    B, T = y_true_mws.shape
    assert len(lead_hours) == T

    lead_to_idx = {h: i for i, h in enumerate(lead_hours)}
    if 0 not in lead_to_idx or horizon_hours not in lead_to_idx:
        raise ValueError(f"lead_hours에 0 또는 {horizon_hours}이(가) 없습니다.")
    i0 = lead_to_idx[0]
    iH = lead_to_idx[horizon_hours]

    valid = (mask[:, i0] > 0.5) & (mask[:, iH] > 0.5)
    valid &= np.isfinite(y_true_mws[:, i0]) & np.isfinite(y_true_mws[:, iH])
    valid &= np.isfinite(y_pred_mws[:, i0]) & np.isfinite(y_pred_mws[:, iH])

    if min_intensity_kt is not None:
        if ts_on == "t0":
            valid &= (y_true_mws[:, i0] >= float(min_intensity_kt))
        elif ts_on == "both":
            valid &= (y_true_mws[:, i0] >= float(min_intensity_kt)) & (y_true_mws[:, iH] >= float(min_intensity_kt))
        else:
            raise ValueError("ts_on은 't0' 또는 'both'만 가능합니다.")

    if np.sum(valid) == 0:
        raise ValueError(
            f"RI/RW 평가에 사용할 유효 샘플이 없습니다. "
            f"(horizon={horizon_hours}h, TS>={min_intensity_kt}kt, ts_on={ts_on})"
        )

    yt0 = y_true_mws[valid, i0]
    ytH = y_true_mws[valid, iH]
    yp0 = y_pred_mws[valid, i0]
    ypH = y_pred_mws[valid, iH]

    d_true = ytH - yt0
    d_pred = ypH - yp0

    def _safe(num, den):
        return float(num) / float(den) if den > 0 else np.nan

    # ================== RI ==================
    true_RI = d_true >= ri_thresh
    pred_RI = d_pred >= ri_thresh
    H_ri  = int(np.sum(true_RI & pred_RI))
    M_ri  = int(np.sum(true_RI & (~pred_RI)))
    F_ri  = int(np.sum((~true_RI) & pred_RI))
    CN_ri = int(np.sum((~true_RI) & (~pred_RI)))
    N_ri  = H_ri + M_ri + F_ri + CN_ri

    POD_ri  = _safe(H_ri, H_ri + M_ri)       # hit rate
    FAR_ri  = _safe(F_ri, H_ri + F_ri)
    CSI_ri  = _safe(H_ri, H_ri + M_ri + F_ri)
    BIAS_ri = _safe(H_ri + F_ri, H_ri + M_ri)
    ACC_ri  = _safe(H_ri + CN_ri, N_ri)

    PREC_ri = _safe(H_ri, H_ri + F_ri)
    F1_ri   = _safe(2 * H_ri, 2 * H_ri + F_ri + M_ri)

    POFD_ri = _safe(F_ri, F_ri + CN_ri)
    TSS_ri  = POD_ri - POFD_ri

    if N_ri > 0:
        H_rand_ri = (H_ri + F_ri) * (H_ri + M_ri) / float(N_ri)
    else:
        H_rand_ri = 0.0
    ETS_ri = _safe(H_ri - H_rand_ri, H_ri + M_ri + F_ri - H_rand_ri)

    denom_hss_ri = ((H_ri + M_ri) * (M_ri + CN_ri) + (H_ri + F_ri) * (F_ri + CN_ri))
    HSS_ri = _safe(2.0 * (H_ri * CN_ri - M_ri * F_ri), denom_hss_ri)

    ri_stats = {
        "event": "RI",
        "horizon_hours": horizon_hours,
        "threshold_kt": float(ri_thresh),
        "min_intensity_kt": float(min_intensity_kt) if min_intensity_kt is not None else None,
        "ts_on": str(ts_on),
        "H": H_ri, "M": M_ri, "F": F_ri, "CN": CN_ri,
        "POD":  POD_ri,
        "FAR":  FAR_ri,
        "CSI":  CSI_ri,
        "BIAS": BIAS_ri,
        "ACC":  ACC_ri,
        "PRECISION": PREC_ri,
        "F1":   F1_ri,
        "POFD": POFD_ri,
        "TSS":  TSS_ri,
        "ETS":  ETS_ri,
        "HSS":  HSS_ri,
    }

    # ================== RW ==================
    true_RW = d_true <= -rw_thresh
    pred_RW = d_pred <= -rw_thresh
    H_rw  = int(np.sum(true_RW & pred_RW))
    M_rw  = int(np.sum(true_RW & (~pred_RW)))
    F_rw  = int(np.sum((~true_RW) & pred_RW))
    CN_rw = int(np.sum((~true_RW) & (~pred_RW)))
    N_rw  = H_rw + M_rw + F_rw + CN_rw

    POD_rw  = _safe(H_rw, H_rw + M_rw)
    FAR_rw  = _safe(F_rw, H_rw + F_rw)
    CSI_rw  = _safe(H_rw, H_rw + M_rw + F_rw)
    BIAS_rw = _safe(H_rw + F_rw, H_rw + M_rw)
    ACC_rw  = _safe(H_rw + CN_rw, N_rw)

    PREC_rw = _safe(H_rw, H_rw + F_rw)
    F1_rw   = _safe(2 * H_rw, 2 * H_rw + F_rw + M_rw)

    POFD_rw = _safe(F_rw, F_rw + CN_rw)
    TSS_rw  = POD_rw - POFD_rw

    if N_rw > 0:
        H_rand_rw = (H_rw + F_rw) * (H_rw + M_rw) / float(N_rw)
    else:
        H_rand_rw = 0.0
    ETS_rw = _safe(H_rw - H_rand_rw, H_rw + M_rw + F_rw - H_rand_rw)

    denom_hss_rw = ((H_rw + M_rw) * (M_rw + CN_rw) + (H_rw + F_rw) * (F_rw + CN_rw))
    HSS_rw = _safe(2.0 * (H_rw * CN_rw - M_rw * F_rw), denom_hss_rw)

    rw_stats = {
        "event": "RW",
        "horizon_hours": horizon_hours,
        "threshold_kt": float(rw_thresh),
        "min_intensity_kt": float(min_intensity_kt) if min_intensity_kt is not None else None,
        "ts_on": str(ts_on),
        "H": H_rw, "M": M_rw, "F": F_rw, "CN": CN_rw,
        "POD":  POD_rw,
        "FAR":  FAR_rw,
        "CSI":  CSI_rw,
        "BIAS": BIAS_rw,
        "ACC":  ACC_rw,
        "PRECISION": PREC_rw,
        "F1":   F1_rw,
        "POFD": POFD_rw,
        "TSS":  TSS_rw,
        "ETS":  ETS_rw,
        "HSS":  HSS_rw,
    }

    return ri_stats, rw_stats

def save_ri_rw_detailed_excel(
    y_true_mws, y_pred_mws, mask, lead_hours, meta_list,
    ri_stats, rw_stats, horizon_hours, save_root,
    ri_thresh=30.0, rw_thresh=30.0,
    min_intensity_kt=34.0,
    ts_on="t0",
):
    y_true_mws = np.asarray(y_true_mws, dtype=np.float32)
    y_pred_mws = np.asarray(y_pred_mws, dtype=np.float32)
    mask       = np.asarray(mask, dtype=np.float32)

    B, T = y_true_mws.shape
    lead_to_idx = {h: i for i, h in enumerate(lead_hours)}
    if 0 not in lead_to_idx or horizon_hours not in lead_to_idx:
        raise ValueError(f"lead_hours에 0 또는 {horizon_hours}h 이(가) 없습니다.")
    i0 = lead_to_idx[0]
    iH = lead_to_idx[horizon_hours]

    rows_ri, rows_rw = [], []

    for b in range(B):
        m = meta_list[b]
        if not (mask[b, i0] > 0.5 and mask[b, iH] > 0.5):
            continue
        if not (
            np.isfinite(y_true_mws[b, i0]) and np.isfinite(y_true_mws[b, iH]) and
            np.isfinite(y_pred_mws[b, i0]) and np.isfinite(y_pred_mws[b, iH])
        ):
            continue

        if min_intensity_kt is not None:
            if ts_on == "t0":
                if not (y_true_mws[b, i0] >= float(min_intensity_kt)):
                    continue
            elif ts_on == "both":
                if not ((y_true_mws[b, i0] >= float(min_intensity_kt)) and (y_true_mws[b, iH] >= float(min_intensity_kt))):
                    continue
            else:
                raise ValueError("ts_on은 't0' 또는 'both'만 가능합니다.")

        t0 = m["t0"]
        basin = m["basin"]
        tid = int(m["id"])
        storm_id = f"{basin}{str(tid).zfill(2)}"

        yt0 = float(y_true_mws[b, i0]); ytH = float(y_true_mws[b, iH])
        yp0 = float(y_pred_mws[b, i0]); ypH = float(y_pred_mws[b, iH])
        d_true = ytH - yt0
        d_pred = ypH - yp0

        # --------- RI per case ----------
        true_ri = d_true >= ri_thresh
        pred_ri = d_pred >= ri_thresh
        if true_ri and pred_ri: cat_ri = "H"
        elif true_ri and (not pred_ri): cat_ri = "M"
        elif (not true_ri) and pred_ri: cat_ri = "F"
        else: cat_ri = "CN"

        rows_ri.append({
            "StormID": storm_id, "Basin": basin, "ID": tid, "T0": t0,
            "HorizonHour": horizon_hours,
            "TS_min_kt": float(min_intensity_kt) if min_intensity_kt is not None else None,
            "TS_on": str(ts_on),
            "True_MWS_0h": yt0, "True_MWS_H": ytH, "Pred_MWS_0h": yp0, "Pred_MWS_H": ypH,
            "True_dMWS": d_true, "Pred_dMWS": d_pred, "AbsErr_dMWS": abs(d_true - d_pred),
            "True_RI": bool(true_ri), "Pred_RI": bool(pred_ri), "RI_Category": cat_ri,
        })

        # --------- RW per case ----------
        true_rw = d_true <= -rw_thresh
        pred_rw = d_pred <= -rw_thresh
        if true_rw and pred_rw: cat_rw = "H"
        elif true_rw and (not pred_rw): cat_rw = "M"
        elif (not true_rw) and pred_rw: cat_rw = "F"
        else: cat_rw = "CN"

        rows_rw.append({
            "StormID": storm_id, "Basin": basin, "ID": tid, "T0": t0,
            "HorizonHour": horizon_hours,
            "TS_min_kt": float(min_intensity_kt) if min_intensity_kt is not None else None,
            "TS_on": str(ts_on),
            "True_MWS_0h": yt0, "True_MWS_H": ytH, "Pred_MWS_0h": yp0, "Pred_MWS_H": ypH,
            "True_dMWS": d_true, "Pred_dMWS": d_pred, "AbsErr_dMWS": abs(d_true - d_pred),
            "True_RW": bool(true_rw), "Pred_RW": bool(pred_rw), "RW_Category": cat_rw,
        })

    df_ri = pd.DataFrame(rows_ri)
    df_rw = pd.DataFrame(rows_rw)

    # --------- Summary sheet: 모든 지표 포함 ----------
    summary_rows = []
    for d in [ri_stats, rw_stats]:
        summary_rows.append({"Event": d["event"], "Metric": "horizon_hours", "Value": d["horizon_hours"]})
        summary_rows.append({"Event": d["event"], "Metric": "threshold_kt", "Value": d["threshold_kt"]})
        summary_rows.append({"Event": d["event"], "Metric": "min_intensity_kt(TS+)", "Value": d.get("min_intensity_kt", None)})
        summary_rows.append({"Event": d["event"], "Metric": "ts_on", "Value": d.get("ts_on", None)})

        for key in [
            "H", "M", "F", "CN",
            "POD", "FAR", "CSI", "BIAS",
            "ACC", "PRECISION", "F1",
            "POFD", "TSS", "ETS", "HSS",
        ]:
            summary_rows.append({"Event": d["event"], "Metric": key, "Value": d.get(key, np.nan)})

    df_summary = pd.DataFrame(summary_rows, columns=["Event", "Metric", "Value"])

    os.makedirs(save_root, exist_ok=True)
    excel_path = os.path.join(save_root, f"contingency_RI_RW_{horizon_hours}h_detail.xlsx")

    engine = None
    try:
        import xlsxwriter  # noqa
        engine = "xlsxwriter"
    except Exception:
        try:
            import openpyxl  # noqa
            engine = "openpyxl"
        except Exception:
            engine = None

    if engine is None:
        base = os.path.splitext(excel_path)[0]
        df_ri.to_csv(base + "_RI_detail.csv", index=False)
        df_rw.to_csv(base + "_RW_detail.csv", index=False)
        df_summary.to_csv(base + "_summary.csv", index=False)
        print(f"⚠️ Excel writer 엔진이 없어 CSV로 저장했습니다:\n  - {base + '_RI_detail.csv'}\n  - {base + '_RW_detail.csv'}\n  - {base + '_summary.csv'}")
        return

    with pd.ExcelWriter(excel_path, engine=engine) as writer:
        df_ri.to_excel(writer, sheet_name="RI_detail", index=False)
        df_rw.to_excel(writer, sheet_name="RW_detail", index=False)
        df_summary.to_excel(writer, sheet_name="Summary", index=False)

    print(f"📄 Saved RI/RW detailed contingency Excel: {excel_path}")


# ================== 4. Utilities (REPLACE) ==================
# ================== 4. Utilities (REPLACE) ==================
def get_background_branches(
    basin, tid, t0, year, local_env_dir, datetimes, max_forecast_hours,
    stats=None, split="", include_prev6=False
):
    """
    ENV sequence builder

    include_prev6=False:
      times = [T, T+6, ..., T+H]               -> length = H/6 + 1
    include_prev6=True:
      times = [T-6, T, T+6, ..., T+H]          -> length = H/6 + 2

    [UPDATED]
    - T-6가 없더라도 샘플을 버리지 않음
    - include_prev6=True일 때 prev6 slot은:
        * 실제 T-6 ENV가 있으면 사용
        * 없으면 T(현재) ENV frame으로 대체
    - 단, 현재(T)는 반드시 존재해야 함
    - target valid mask는 여전히 [0,+6,+12,...,+H] 기준
    - 최소 +6, +12까지 실제 track/ENV가 있어야 샘플 사용
    """
    if stats is None:
        stats = defaultdict(int)

    tid = str(tid).zfill(2)
    result = {}

    if include_prev6:
        hours = [-6, 0] + list(range(6, max_forecast_hours + 1, 6))
    else:
        hours = list(range(0, max_forecast_hours + 1, 6))

    expected_times = [t0 + timedelta(hours=h) for h in hours]
    has_track = [(tt in datetimes) for tt in expected_times]

    # --------------------------------------------------
    # 현재(T)는 반드시 있어야 함
    # prev6는 없어도 허용
    # --------------------------------------------------
    if include_prev6:
        if len(has_track) < 2 or (not has_track[1]):
            stats["missing_curr_track"] += 1
            _vprint(f"[MISS][{split}] current track missing: tid={basin}{tid} t0={t0}")
            return None, None
    else:
        if len(has_track) < 1 or (not has_track[0]):
            stats["missing_curr_track"] += 1
            _vprint(f"[MISS][{split}] current track missing: tid={basin}{tid} t0={t0}")
            return None, None

    # --------------------------------------------------
    # target 기준 env valid mask 계산
    # target axis: [0,+6,+12,...,+H]
    # 실제 미래 ENV가 연속적으로 존재하는 마지막 리드까지만 1
    # --------------------------------------------------
    target_hours = list(range(0, max_forecast_hours + 1, 6))
    env_valid_mask_target = np.zeros((len(target_hours),), dtype=np.float32)
    env_valid_mask_target[0] = 1.0  # current는 반드시 존재

    def _env_file_exists_for_all_vars(tt):
        dt_str = tt.strftime("%Y%m%d%H")

        for var, levels in multi_level_dict.items():
            for lv in levels:
                path = os.path.join(local_env_dir, f"{var}{lv}.{basin}{tid}.{dt_str}")
                if not os.path.exists(path):
                    return False

        for var in ["SST", "TOPO"]:
            path = os.path.join(local_env_dir, f"{var}.{basin}{tid}.{dt_str}")
            if not os.path.exists(path):
                return False

        return True

    for j, h in enumerate(target_hours[1:], start=1):
        tt = t0 + timedelta(hours=h)
        if (tt in datetimes) and _env_file_exists_for_all_vars(tt):
            env_valid_mask_target[j] = 1.0
        else:
            break

    # 최소 +6, +12까지는 실제로 있어야 샘플 사용
    if len(env_valid_mask_target) > 2:
        if not (env_valid_mask_target[1] > 0.5 and env_valid_mask_target[2] > 0.5):
            stats["miss_track_steps"] += 1
            _vprint(f"[MISS][{split}] future ENV too short: tid={basin}{tid} t0={t0}")
            return None, None

    # --------------------------------------------------
    # helper: 현재 시점 frame 읽기
    # --------------------------------------------------
    def _load_env_frame(var, lv, tt):
        dt_str = tt.strftime("%Y%m%d%H")
        if lv is None:
            path = os.path.join(local_env_dir, f"{var}.{basin}{tid}.{dt_str}")
        else:
            path = os.path.join(local_env_dir, f"{var}{lv}.{basin}{tid}.{dt_str}")

        if not os.path.exists(path):
            return None

        bg = load_background(path, crop=True)
        if bg is None:
            return None
        return bg.astype(np.float32)

    # --------------------------------------------------
    # 현재(T) ENV를 먼저 확보
    # prev6 없으면 이걸 복사해서 prev6 slot에 넣음
    # --------------------------------------------------
    current_env_cache = {}

    for var, levels in multi_level_dict.items():
        current_env_cache[var] = {}
        for lv in levels:
            cur_frame = _load_env_frame(var, lv, t0)
            if cur_frame is None:
                stats["missing_curr_env"] += 1
                _vprint(f"[MISS][{split}] current ENV missing: var={var} level={lv} t0={t0}")
                return None, None
            current_env_cache[var][lv] = cur_frame

    for var in ["SST", "TOPO"]:
        cur_frame = _load_env_frame(var, None, t0)
        if cur_frame is None:
            stats["missing_curr_env"] += 1
            _vprint(f"[MISS][{split}] current ENV missing: var={var} t0={t0}")
            return None, None
        current_env_cache[var] = cur_frame

    # --------------------------------------------------
    # 실제 tensor 구성
    # - prev6 없으면 current frame 복사
    # - 미래 없으면 직전 frame 복사 (기존과 동일)
    # --------------------------------------------------
    for var, levels in multi_level_dict.items():
        frames_per_level = []
        for lv in levels:
            frames = []
            last_frame = None

            for h, t_needed, ok in zip(hours, expected_times, has_track):
                # prev6 special handling
                if include_prev6 and h == -6:
                    if ok:
                        bg = _load_env_frame(var, lv, t_needed)
                        if bg is None:
                            # prev6 track은 있지만 env file이 없으면 current 복사
                            bg = current_env_cache[var][lv].copy()
                        last_frame = bg.astype(np.float32)
                        frames.append(last_frame)
                    else:
                        bg = current_env_cache[var][lv].copy()
                        last_frame = bg.astype(np.float32)
                        frames.append(last_frame)
                    continue

                # current / future
                if ok:
                    bg = _load_env_frame(var, lv, t_needed)
                    if bg is not None:
                        last_frame = bg.astype(np.float32)
                        frames.append(last_frame)
                    else:
                        # current는 반드시 있어야 함
                        if h == 0:
                            stats["missing_curr_env"] += 1
                            _vprint(f"[MISS][{split}] current ENV missing: var={var} level={lv} t0={t0}")
                            return None, None
                        if last_frame is None:
                            stats["miss_track_steps"] += 1
                            _vprint(f"[MISS][{split}] no valid ENV frame yet: tid={basin}{tid} t0={t0}")
                            return None, None
                        frames.append(last_frame.copy())
                else:
                    if h == 0:
                        stats["missing_curr_env"] += 1
                        _vprint(f"[MISS][{split}] current track/env missing: var={var} level={lv} t0={t0}")
                        return None, None
                    if last_frame is None:
                        stats["miss_track_steps"] += 1
                        _vprint(f"[MISS][{split}] no valid ENV frame yet: tid={basin}{tid} t0={t0}")
                        return None, None
                    frames.append(last_frame.copy())

            frames_per_level.append(np.stack(frames, axis=0))

        result[var] = np.stack(frames_per_level, axis=1)  # (Tin,L,H,W)

    for var in ["SST", "TOPO"]:
        seq = []
        last_frame = None

        for h, t_needed, ok in zip(hours, expected_times, has_track):
            if include_prev6 and h == -6:
                if ok:
                    arr = _load_env_frame(var, None, t_needed)
                    if arr is None:
                        arr = current_env_cache[var].copy()
                    last_frame = arr.astype(np.float32)
                    seq.append(last_frame)
                else:
                    arr = current_env_cache[var].copy()
                    last_frame = arr.astype(np.float32)
                    seq.append(last_frame)
                continue

            if ok:
                arr = _load_env_frame(var, None, t_needed)
                if arr is not None:
                    last_frame = arr.astype(np.float32)
                    seq.append(last_frame)
                else:
                    if h == 0:
                        stats["missing_curr_env"] += 1
                        _vprint(f"[MISS][{split}] current ENV missing: var={var} t0={t0}")
                        return None, None
                    if last_frame is None:
                        stats["miss_track_steps"] += 1
                        _vprint(f"[MISS][{split}] no valid {var} frame yet: tid={basin}{tid} t0={t0}")
                        return None, None
                    seq.append(last_frame.copy())
            else:
                if h == 0:
                    stats["missing_curr_env"] += 1
                    _vprint(f"[MISS][{split}] current track/env missing: var={var} t0={t0}")
                    return None, None
                if last_frame is None:
                    stats["miss_track_steps"] += 1
                    _vprint(f"[MISS][{split}] no valid {var} frame yet: tid={basin}{tid} t0={t0}")
                    return None, None
                seq.append(last_frame.copy())

        result[var] = np.stack(seq, axis=0)[:, None, :, :]  # (Tin,1,H,W)

    return result, env_valid_mask_target

@tf.keras.utils.register_keras_serializable(package="Custom")
def physical_loss(y_true, y_pred):
    """
    [REVISED]
    - MWS 중심 loss
    - MWS: AMP 적용
    - MSLP: AMP 미적용
    - 최종 비중: MWS 0.8 / MSLP 0.2

    y_true: (B, T, 5)
      [:, 1:, 0] = z-scored cumulative delta for MWS
      [:, 1:, 1] = z-scored cumulative delta for MSLP
      [:, 1:, 2] = cumulative-valid mask
      [:, 1:, 3] = AMP for MWS
      [:, 1:, 4] = AMP for MSLP (현재 loss에서는 사용 안 함)

    y_pred: (B, T-1, 2)
      [:, :, 0] = predicted z-scored cumulative delta for MWS
      [:, :, 1] = predicted z-scored cumulative delta for MSLP
    """
    eps = tf.constant(1e-6, dtype=tf.float32)
    huber_delta = tf.constant(2.0, dtype=tf.float32)

    yt   = tf.cast(y_true[:, 1:, 0:2], tf.float32)   # (B, T-1, 2)
    mask = tf.cast(y_true[:, 1:, 2],   tf.float32)   # (B, T-1)
    amp_mws = tf.cast(y_true[:, 1:, 3], tf.float32)  # (B, T-1)

    yp = tf.cast(y_pred, tf.float32)                 # (B, T-1, 2)

    # finite safety
    yt      = tf.where(tf.math.is_finite(yt), yt, 0.0)
    yp      = tf.where(tf.math.is_finite(yp), yp, 0.0)
    mask    = tf.where(tf.math.is_finite(mask), mask, 0.0)
    amp_mws = tf.where(tf.math.is_finite(amp_mws), amp_mws, 1.0)

    # -----------------------------
    # MWS loss (AMP 적용)
    # -----------------------------
    err_mws = yt[:, :, 0] - yp[:, :, 0]
    abs_err_mws = tf.abs(err_mws)

    quad_mws = tf.minimum(abs_err_mws, huber_delta)
    lin_mws  = abs_err_mws - quad_mws
    huber_mws = 0.5 * tf.square(quad_mws) + huber_delta * lin_mws  # (B, T-1)

    huber_mws = huber_mws * amp_mws
    num_mws = tf.reduce_sum(huber_mws * mask)
    den_mws = tf.reduce_sum(mask) + eps
    loss_mws = num_mws / den_mws

    # -----------------------------
    # MSLP loss (AMP 미적용)
    # -----------------------------
    err_mslp = yt[:, :, 1] - yp[:, :, 1]
    abs_err_mslp = tf.abs(err_mslp)

    quad_mslp = tf.minimum(abs_err_mslp, huber_delta)
    lin_mslp  = abs_err_mslp - quad_mslp
    huber_mslp = 0.5 * tf.square(quad_mslp) + huber_delta * lin_mslp  # (B, T-1)

    num_mslp = tf.reduce_sum(huber_mslp * mask)
    den_mslp = tf.reduce_sum(mask) + eps
    loss_mslp = num_mslp / den_mslp

    # -----------------------------
    # Final: MWS 중심
    # -----------------------------
    return 0.8 * loss_mws + 0.2 * loss_mslp

# ================== Flags / Cache ==================
USE_IR = True
_PREPROC_CACHE = {}

# ================== (REWRITE) 5. Sample Creation (IRWVLN optional) ==================
# ================== (REWRITE) 5. Sample Creation (IRWVLN optional) ==================
def create_samples(df, split, local_env_dir, max_forecast_hours):
    """
    [UPDATED: allow missing T-6]

    입력:
      - state at T-6, T:
          IR(T-6), IR(T)
          AUX(T-6), AUX(T)
          ENV(T-6), ENV(T)
      - future forcing:
          ENV(T+6), T+12, ..., T+H

    타깃:
      - cumulative delta from T
          MWS(T+h)  - MWS(T)
          MSLP(T+h) - MSLP(T)

    [핵심 변경]
      - T-6 track이 없어도 샘플 사용
      - prev6 정보가 없으면:
          * AUX prev6 slot = current 값 복사
          * dmws6/dmslp6/rel_dmws6 = 0
          * has_prev6 = 0
          * IR(T-6) = IR(T) 복사
          * ENV(T-6) = get_background_branches에서 T frame 복사
      - 단, T / +6 / +12는 반드시 살아있어야 함
    """
    TARGET_H = int(GRID["TARGET_H"])
    TARGET_W = int(GRID["TARGET_W"])
    ENV_SPAN_DEG = float(GRID["ENV_SPAN_DEG"])
    IR_RES_DEG = float(GRID["IR_NATIVE_DEG"])

    REL_MWS_DENOM_MIN = 10.0

    T = max_forecast_hours // 6 + 1
    Tin = T + 1
    lead_hours = list(range(0, max_forecast_hours + 1, 6))
    expected_offsets = [timedelta(hours=h) for h in lead_hours]

    X_env = []
    X_aux = []
    X_ir_crop = []

    y_abs_obs_mws_list = []
    y_abs_obs_mslp_list = []
    y_abs_ffill_mws_list = []
    y_abs_ffill_mslp_list = []
    mask_abs_list = []
    env_valid_mask_list = []
    meta_list = []

    if df is None or len(df) == 0:
        return (
            {}, np.array([]), np.array([]),
            np.array([]), np.array([]),
            np.array([]), np.array([]),
            np.array([]), np.array([]), []
        )

    required_cols = ["datetime", "latitude", "longitude", "mws", "mslp"]
    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"[create_samples] required column missing: '{c}'.")

    df = df.copy().dropna(subset=required_cols).copy()
    grouped = df.groupby(["basin", "id"], sort=False)

    for (basin, tid_raw), df_tc in tqdm(grouped, desc=f"{split} samples(cum-delta | state[T-6,T] + future ENV)"):

        df_tc = (
            df_tc.sort_values("datetime")
                 .drop_duplicates(subset=["datetime"], keep="first")
                 .copy()
        )
        if len(df_tc) == 0:
            continue

        try:
            tid = int(tid_raw)
        except Exception:
            tid = tid_raw

        dts       = pd.to_datetime(df_tc["datetime"]).to_list()
        lat_vals  = df_tc["latitude"].astype(float).to_numpy()
        lon_vals  = df_tc["longitude"].astype(float).to_numpy()
        mws_vals  = df_tc["mws"].astype(float).to_numpy()
        mslp_vals = df_tc["mslp"].astype(float).to_numpy()

        dt_map = {
            pd.to_datetime(dt): (float(lat), float(lon), float(mws), float(mslp))
            for dt, lat, lon, mws, mslp in zip(dts, lat_vals, lon_vals, mws_vals, mslp_vals)
        }
        dts_set = set(dt_map.keys())

        for t0, lat0, lon0, mws0, mslp0 in df_tc[["datetime", "latitude", "longitude", "mws", "mslp"]].itertuples(index=False, name=None):
            t0 = pd.to_datetime(t0)
            lat0 = float(lat0); lon0 = float(lon0)
            mws0 = float(mws0); mslp0 = float(mslp0)

            if not (np.isfinite(lat0) and np.isfinite(lon0) and np.isfinite(mws0) and np.isfinite(mslp0)):
                continue
            if mws0 < 5.0:
                continue

            if lon0 > 180.0:
                lon0 -= 360.0

            t_m6 = t0 - timedelta(hours=6)
            has_prev6 = 1.0 if (t_m6 in dt_map) else 0.0

            if has_prev6 > 0.5:
                lat_m6, lon_m6, mws_m6, mslp_m6 = dt_map[t_m6]
                lat_m6 = float(lat_m6)
                lon_m6 = float(lon_m6)
                mws_m6 = float(mws_m6)
                mslp_m6 = float(mslp_m6)
                if lon_m6 > 180.0:
                    lon_m6 -= 360.0

                dmws6  = float(mws0 - mws_m6)
                dmslp6 = float(mslp0 - mslp_m6)
                rel_dmws6 = float(dmws6) / max(float(mws0), float(REL_MWS_DENOM_MIN))
            else:
                # prev6 없으면 current 복사 + 변화량 0 + flag=0
                lat_m6 = float(lat0)
                lon_m6 = float(lon0)
                mws_m6 = float(mws0)
                mslp_m6 = float(mslp0)
                dmws6 = 0.0
                dmslp6 = 0.0
                rel_dmws6 = 0.0

            # 최소 +6, +12 관측 존재
            if not ((t0 + timedelta(hours=6)) in dts_set and (t0 + timedelta(hours=12)) in dts_set):
                continue

            bg, env_valid_mask = get_background_branches(
                basin=basin, tid=tid, t0=t0, year=t0.year,
                local_env_dir=local_env_dir, datetimes=dts_set,
                max_forecast_hours=max_forecast_hours,
                stats=None, split=split, include_prev6=True
            )
            if bg is None or env_valid_mask is None:
                continue

            env_ok = True
            for k, arr in bg.items():
                if not isinstance(arr, np.ndarray) or arr.ndim != 4:
                    env_ok = False
                    break
                if arr.shape[0] != Tin:
                    env_ok = False
                    break
                if arr.shape[-2] != TARGET_H or arr.shape[-1] != TARGET_W:
                    env_ok = False
                    break
            if not env_ok:
                continue

            if env_valid_mask.shape[0] != T:
                continue

            if "SST" in bg:
                sst = bg["SST"].astype(np.float32)
                if "TOPO" in bg:
                    topo_raw = bg["TOPO"].astype(np.float32)
                    land0_t = topo_raw[:, 0] >= 10.0
                    land_mask_local = land0_t
                else:
                    land_mask_local = np.zeros_like(sst[:, 0], dtype=bool)

                finite = np.isfinite(sst)
                phys_range = (sst >= 250.0) & (sst <= 400.0)
                invalid_any = (~finite) | (~phys_range) | land_mask_local[:, None, :, :]
                sst[invalid_any] = np.nan
                bg["SST"] = sst

            # --------------------------------------------------
            # IR pair
            # - current IR는 반드시 필요
            # - prev6 IR가 없으면 current IR 복사
            # --------------------------------------------------
            ir_pair = []

            dt_str0 = t0.strftime("%Y%m%d%H")
            ir_path0 = os.path.join(local_env_dir, f"IRWVLN.{basin}{str(tid).zfill(2)}.{dt_str0}")
            if not os.path.exists(ir_path0):
                continue

            ir_full0 = load_irwvln(ir_path0)
            if ir_full0 is None:
                continue

            finite_mask0 = np.isfinite(ir_full0)
            if not np.any(finite_mask0):
                continue

            IR_MIN_PHYS, IR_MAX_PHYS = -15.0, 30.0
            min_val0 = float(np.nanmin(ir_full0[finite_mask0]))
            max_val0 = float(np.nanmax(ir_full0[finite_mask0]))
            if (min_val0 < IR_MIN_PHYS) or (max_val0 > IR_MAX_PHYS):
                continue

            ir_curr = center_crop_or_resize_by_grid(
                arr2d=ir_full0,
                src_res_deg=IR_RES_DEG,
                target_hw=TARGET_H,
                span_deg=ENV_SPAN_DEG
            )
            if ir_curr is None or ir_curr.shape != (TARGET_H, TARGET_W):
                continue

            ir_prev = None
            if has_prev6 > 0.5:
                dt_strm6 = t_m6.strftime("%Y%m%d%H")
                ir_pathm6 = os.path.join(local_env_dir, f"IRWVLN.{basin}{str(tid).zfill(2)}.{dt_strm6}")
                if os.path.exists(ir_pathm6):
                    ir_fullm6 = load_irwvln(ir_pathm6)
                    if ir_fullm6 is not None:
                        finite_maskm6 = np.isfinite(ir_fullm6)
                        if np.any(finite_maskm6):
                            min_valm6 = float(np.nanmin(ir_fullm6[finite_maskm6]))
                            max_valm6 = float(np.nanmax(ir_fullm6[finite_maskm6]))
                            if (min_valm6 >= IR_MIN_PHYS) and (max_valm6 <= IR_MAX_PHYS):
                                ir_prev_try = center_crop_or_resize_by_grid(
                                    arr2d=ir_fullm6,
                                    src_res_deg=IR_RES_DEG,
                                    target_hw=TARGET_H,
                                    span_deg=ENV_SPAN_DEG
                                )
                                if ir_prev_try is not None and ir_prev_try.shape == (TARGET_H, TARGET_W):
                                    ir_prev = ir_prev_try

            if ir_prev is None:
                ir_prev = ir_curr.copy()

            ir_pair.append(ir_prev[..., None].astype(np.float32))
            ir_pair.append(ir_curr[..., None].astype(np.float32))

            times = [t0 + off for off in expected_offsets]

            y_obs_mws  = np.full((T,), np.nan, dtype=np.float32)
            y_obs_mslp = np.full((T,), np.nan, dtype=np.float32)
            mask_abs   = np.zeros((T,), dtype=np.float32)

            y_obs_mws[0] = mws0
            y_obs_mslp[0] = mslp0
            mask_abs[0] = 1.0

            for j in range(1, T):
                tj = times[j]
                if tj in dt_map:
                    _, _, mws_j, mslp_j = dt_map[tj]
                    y_obs_mws[j]  = float(mws_j)
                    y_obs_mslp[j] = float(mslp_j)
                    mask_abs[j]   = 1.0

            y_ff_mws  = y_obs_mws.copy()
            y_ff_mslp = y_obs_mslp.copy()
            for j in range(1, T):
                if not np.isfinite(y_ff_mws[j]):
                    y_ff_mws[j] = y_ff_mws[j-1]
                if not np.isfinite(y_ff_mslp[j]):
                    y_ff_mslp[j] = y_ff_mslp[j-1]

            X_env.append(bg)
            X_ir_crop.append(np.stack(ir_pair, axis=0).astype(np.float32))
            X_aux.append(np.asarray([
                lat_m6, lon_m6, mws_m6, mslp_m6,
                lat0,   lon0,   mws0,   mslp0,
                dmws6, dmslp6, rel_dmws6, has_prev6
            ], dtype=np.float32))

            y_abs_obs_mws_list.append(y_obs_mws.astype(np.float32))
            y_abs_obs_mslp_list.append(y_obs_mslp.astype(np.float32))
            y_abs_ffill_mws_list.append(y_ff_mws.astype(np.float32))
            y_abs_ffill_mslp_list.append(y_ff_mslp.astype(np.float32))
            mask_abs_list.append(mask_abs.astype(np.float32))
            env_valid_mask_list.append(env_valid_mask.astype(np.float32))

            meta_list.append({
                "basin": basin, "id": tid, "t0": t0,
                "lat_m6": float(lat_m6), "lon_m6": float(lon_m6),
                "lat0": float(lat0), "lon0": float(lon0),
                "mws_m6": float(mws_m6), "mslp_m6": float(mslp_m6),
                "mws0": float(mws0), "mslp0": float(mslp0),
                "dmws6": float(dmws6), "dmslp6": float(dmslp6),
                "rel_dmws6": float(rel_dmws6),
                "has_prev6": float(has_prev6),
                "grid_env_res_deg": float(GRID["ENV_RES_DEG"]),
                "grid_ir_res_deg": float(GRID["IR_NATIVE_DEG"]),
                "grid_span_deg": float(GRID["ENV_SPAN_DEG"]),
                "grid_target_hw": int(GRID["TARGET_H"]),
                "input_time_layout": "[-6, 0, +6, ..., +H]",
                "target_layout": "[0, +6, ..., +H]",
            })

    if len(X_env) == 0:
        return (
            {}, np.array([]), np.array([]),
            np.array([]), np.array([]),
            np.array([]), np.array([]),
            np.array([]), np.array([]), []
        )

    env_keys = X_env[0].keys()
    X_env_dict = {k: np.stack([x[k] for x in X_env], axis=0).astype(np.float32) for k in env_keys}

    X_aux     = np.stack(X_aux, axis=0).astype(np.float32)
    X_ir_crop = np.stack(X_ir_crop, axis=0).astype(np.float32)

    y_abs_obs_mws    = np.stack(y_abs_obs_mws_list,    axis=0).astype(np.float32)
    y_abs_obs_mslp   = np.stack(y_abs_obs_mslp_list,   axis=0).astype(np.float32)
    y_abs_ffill_mws  = np.stack(y_abs_ffill_mws_list,  axis=0).astype(np.float32)
    y_abs_ffill_mslp = np.stack(y_abs_ffill_mslp_list, axis=0).astype(np.float32)
    mask_abs         = np.stack(mask_abs_list,         axis=0).astype(np.float32)
    env_valid_mask   = np.stack(env_valid_mask_list,   axis=0).astype(np.float32)

    return (
        X_env_dict, X_aux, X_ir_crop,
        y_abs_obs_mws, y_abs_obs_mslp,
        y_abs_ffill_mws, y_abs_ffill_mslp,
        mask_abs, env_valid_mask, meta_list
    )

# ================== 6. Normalization ==================
def normalize_features(X_env, X_ir_crop, scalers=None, fit=True, eps=1e-6, augment_global=None):
    """
    ENV:
      - sequence length Tin = [-6, 0, +6, ..., +H] 어떤 길이든 처리
      - 규칙은 기존과 동일 (TOPO/SST/per-level vars)
    IR:
      - 이제 shape (B, 2, H, W, 1)
      - 두 프레임 전체에 대해 train global z-score
    """
    X_out, new_scalers = {}, {}

    # ---------- 1) TOPO ----------
    if "TOPO" not in X_env:
        raise ValueError("[normalize_features] X_env must contain 'TOPO'.")

    topo = X_env["TOPO"].astype(np.float32)  # (B,Tin,1,H,W)
    topo = np.clip(topo, 0.0, 8000.0)

    B, Tin, _, H, W = topo.shape

    if fit:
        LAND_THRESH_M = 10.0
        TOPO_REF_M    = LAND_THRESH_M
        std_land      = None
    else:
        if scalers is None or "TOPO" not in scalers:
            raise ValueError("[normalize_features] fit=False인데 scalers['TOPO']가 없습니다.")
        LAND_THRESH_M = float(scalers["TOPO"].get("land_thresh_m", 10.0))
        TOPO_REF_M    = float(scalers["TOPO"].get("mu", LAND_THRESH_M))
        std_land      = float(scalers["TOPO"].get("std", 1.0))
        if (not np.isfinite(std_land)) or std_land < 1e-8:
            std_land = 1.0

    # t=1(current) 기준 land-mask를 우선 사용, 없으면 t=0(prev6) fallback
    ref_idx = 1 if Tin > 1 else 0
    land0 = (topo[:, ref_idx, 0] >= LAND_THRESH_M)  # (B,H,W)

    land0_dn = np.empty_like(land0, dtype=bool)
    for b in range(B):
        m = land0[b].astype(np.uint8)
        m = cv2.medianBlur(m, 3)
        land0_dn[b] = (m > 0)
    land0 = land0_dn

    land_mask = np.broadcast_to(land0[:, None, None, :, :], (B, Tin, 1, H, W))

    if fit:
        vals = topo[:, ref_idx, 0][land0]
        vals = vals[np.isfinite(vals)]

        if vals.size > 0:
            std_land = float(np.std(vals - TOPO_REF_M))
        else:
            std_land = 1.0

        if (not np.isfinite(std_land)) or std_land < 1e-8:
            std_land = 1.0

        new_scalers["TOPO"] = {
            "mode": "land_only_z_ref10m",
            "mu": float(TOPO_REF_M),
            "std": float(std_land),
            "land_thresh_m": float(LAND_THRESH_M),
        }

    topo_z = np.where(
        land_mask,
        (topo - TOPO_REF_M) / (std_land + eps),
        0.0
    ).astype(np.float32)
    topo_z = np.clip(topo_z, -5.0, 5.0, out=topo_z)
    topo_z[~np.isfinite(topo_z)] = 0.0
    X_out["TOPO"] = topo_z

    # ---------- 2) SST ----------
    SST_SENTINEL_Z = -10.0
    if "SST" in X_env:
        sst = X_env["SST"].astype(np.float32)  # (B,Tin,1,H,W)

        finite = np.isfinite(sst)
        phys   = (sst >= 250.0) & (sst <= 400.0)
        ocean_valid = finite & phys & (~land_mask)

        K2C = 273.15

        if fit:
            if np.any(ocean_valid):
                sst_c_vals = (sst[ocean_valid] - K2C)
                mu_c  = float(np.mean(sst_c_vals))
                std_c = float(np.std(sst_c_vals))
            else:
                mu_c, std_c = 26.0, 1.0

            if not np.isfinite(mu_c):
                mu_c = 26.0
            if (not np.isfinite(std_c)) or std_c < 1e-8:
                std_c = 1.0

            new_scalers["SST"] = {
                "mu_C": float(mu_c),
                "std_C": float(std_c),
                "land_thresh_m": float(LAND_THRESH_M),
                "sentinel_z": float(SST_SENTINEL_Z),
                "mode": "ocean_only_z_with_land_sentinel"
            }
        else:
            if scalers is None or "SST" not in scalers:
                raise ValueError("[normalize_features] fit=False인데 scalers['SST']가 없습니다.")
            mu_c  = float(scalers["SST"].get("mu_C", 26.0))
            std_c = float(scalers["SST"].get("std_C", 1.0))
            if not np.isfinite(mu_c):
                mu_c = 26.0
            if (not np.isfinite(std_c)) or std_c < 1e-8:
                std_c = 1.0
            SST_SENTINEL_Z = float(scalers["SST"].get("sentinel_z", SST_SENTINEL_Z))

        sst_z = np.full_like(sst, fill_value=SST_SENTINEL_Z, dtype=np.float32)

        if np.any(ocean_valid):
            sst_c_all = (sst - K2C)
            tmp = (sst_c_all - mu_c) / (std_c + eps)
            tmp = np.clip(tmp, -5.0, 5.0)
            sst_z[ocean_valid] = tmp[ocean_valid]

        sst_z[~np.isfinite(sst_z)] = SST_SENTINEL_Z
        X_out["SST"] = sst_z

    # ---------- 3) 3D vars: per-level ----------
    for var in ["ERH", "EDIVE", "EVORT", "EVWS", "ETHETA"]:
        if var not in X_env:
            continue

        arr = X_env[var].astype(np.float32)  # (B,Tin,L,H,W)
        if arr.ndim != 5:
            raise ValueError(f"[normalize_features] X_env['{var}'] shape must be (B,Tin,L,H,W), got {arr.shape}")

        Bv, Tv, L, Hv, Wv = arr.shape

        if fit:
            mu_l  = np.zeros((L,), dtype=np.float32)
            std_l = np.ones((L,), dtype=np.float32)

            for l in range(L):
                vals = arr[:, :, l, :, :].reshape(-1)
                vals = vals[np.isfinite(vals)]
                if vals.size == 0:
                    mu, sd = 0.0, 1.0
                else:
                    mu = float(np.mean(vals))
                    sd = float(np.std(vals))
                    if not np.isfinite(mu):
                        mu = 0.0
                    if (not np.isfinite(sd)) or sd < 1e-8:
                        sd = 1.0
                mu_l[l]  = mu
                std_l[l] = sd

            new_scalers[var] = {
                "mode": "per_level",
                "mu": mu_l.astype(np.float32),
                "std": std_l.astype(np.float32),
                "L_original": int(L),
            }
        else:
            if scalers is None or var not in scalers:
                raise ValueError(f"[normalize_features] fit=False인데 scalers['{var}']가 없습니다.")
            sc = scalers[var]
            if sc.get("mode", None) != "per_level":
                raise ValueError(f"[normalize_features] scalers['{var}']['mode'] must be 'per_level'.")

            mu_l  = np.asarray(sc["mu"], dtype=np.float32)
            std_l = np.asarray(sc["std"], dtype=np.float32)

            L_orig = int(sc.get("L_original", mu_l.shape[0] if mu_l.ndim == 1 else -1))
            if mu_l.ndim != 1 or std_l.ndim != 1:
                raise ValueError(f"[normalize_features] {var} scaler mu/std must be 1D (L,), got mu={mu_l.shape}, std={std_l.shape}")

            if (L_orig != L) or (mu_l.shape[0] != L) or (std_l.shape[0] != L):
                raise ValueError(
                    f"[normalize_features] {var} level 수(L) 불일치: scaler L={L_orig} mu.shape={mu_l.shape} std.shape={std_l.shape}, arr L={L}"
                )

            std_l = np.where((~np.isfinite(std_l)) | (std_l < 1e-8), 1.0, std_l).astype(np.float32)
            mu_l[~np.isfinite(mu_l)] = 0.0

        mu_b  = mu_l[None, None, :, None, None]
        std_b = std_l[None, None, :, None, None]

        z = (arr - mu_b) / (std_b + eps)
        z = np.clip(z, -5.0, 5.0)
        z[~np.isfinite(z)] = 0.0
        X_out[var] = z

    # ---------- 4) IRWVLN pair ----------
    if X_ir_crop.ndim != 5:
        raise ValueError(f"[normalize_features] X_ir_crop must be (B,2,H,W,1), got {X_ir_crop.shape}")

    ir_raw = X_ir_crop.astype(np.float32)[..., 0]  # (B,2,H,W)
    IR_MIN, IR_MAX = -15.0, 30.0

    if fit:
        mask_valid = np.isfinite(ir_raw) & (ir_raw >= IR_MIN) & (ir_raw <= IR_MAX)
        if np.any(mask_valid):
            vals = ir_raw[mask_valid]
            mu_ir  = float(np.mean(vals))
            std_ir = float(np.std(vals))
        else:
            mu_ir, std_ir = 0.0, 1.0
        if not np.isfinite(mu_ir):
            mu_ir = 0.0
        if (not np.isfinite(std_ir)) or std_ir < 1e-8:
            std_ir = 1.0

        new_scalers["IR_crop"] = {
            "mode": "zscore_pair",
            "mu": mu_ir,
            "std": std_ir,
            "clip": (float(IR_MIN), float(IR_MAX)),
        }
    else:
        if scalers is None or "IR_crop" not in scalers:
            raise ValueError("[normalize_features] fit=False인데 scalers['IR_crop']가 없습니다.")
        mu_ir  = float(scalers["IR_crop"].get("mu", 0.0))
        std_ir = float(scalers["IR_crop"].get("std", 1.0))
        if not np.isfinite(mu_ir):
            mu_ir = 0.0
        if (not np.isfinite(std_ir)) or std_ir < 1e-8:
            std_ir = 1.0
        IR_MIN, IR_MAX = scalers["IR_crop"].get("clip", (IR_MIN, IR_MAX))

    ir_clip = np.clip(ir_raw, IR_MIN, IR_MAX)
    ir_z = (ir_clip - mu_ir) / (std_ir + eps)
    ir_z = np.clip(ir_z, -5.0, 5.0)
    ir_z[~np.isfinite(ir_z)] = 0.0
    X_ir_out = ir_z[..., None].astype(np.float32)  # (B,2,H,W,1)

    return X_out, X_ir_out, (new_scalers if fit else scalers)

# ================== 7. Model (FULL REVISION: ABS ENV + STEP-DELTA) ==================
# Key changes:
#  (1) SpatialAttentionPooling2D is applied directly to (B,T,H,W,C) output of ConvLSTM
#      -> removed TimeDistributed(SpatialAttentionPooling2D) duplication risk.
#  (2) physical_loss now (optionally) applies LEAD_WEIGHTS_TENSOR (per-lead weighting) if provided.
#  (3) Custom layer output spec hooks removed to reduce Keras3 trace/compat risk; keep rank-4/5 support.

def autoscale_widths_from_dec_filters(dec_filters: int, scale: int = 1):
    df = int(dec_filters)
    df2 = max(df // 2, 4)

    env_latent_dim = int(round(df * 2.0))
    env_latent_dim = int(max(48, min(env_latent_dim, 128)))

    fused_channels = int(round(df * 2.8))
    fused_channels = int(max(64, min(fused_channels, 192)))

    base = 12.0
    step = 2
    min_heads = 2
    max_heads = 16

    attn_heads = int(step * math.floor((df / base) + 0.5))
    attn_heads = max(min_heads, min(max_heads, attn_heads))
    if attn_heads % 2 != 0:
        attn_heads = min(max_heads, attn_heads + 1)

    attn_keydim = 32
    mha_width = attn_heads * attn_keydim

    mlp_dim = int(round(max(2.0 * df, 1.5 * mha_width)))
    mlp_dim = int(max(192, min(mlp_dim, 512)))

    print(
        f"[HP->WIDTHS] dec_filters={df} scale={scale} | "
        f"env_latent_dim={env_latent_dim} fused_channels={fused_channels} | "
        f"MHA heads={attn_heads}, keydim={attn_keydim} (width={mha_width}) | "
        f"MLP={mlp_dim} | ConvLSTM df/df2={df}/{df2}"
    )

    return {
        "df": df, "df2": df2,
        "env_latent_dim": env_latent_dim,
        "fused_channels": fused_channels,
        "attn_heads": attn_heads,
        "attn_keydim": attn_keydim,
        "mlp_dim": mlp_dim,
    }

# ---- Custom Layers ----
@tf.keras.utils.register_keras_serializable(package="Custom")
class SpatialAttentionPooling2D(tf.keras.layers.Layer):
    """
    Spatial Attention Pooling
      - Input  (B,H,W,C)   -> Output (B,C)
      - Input  (B,T,H,W,C) -> Output (B,T,C)

    1x1 conv -> logits -> softmax -> weighted sum pooling
    """
    def __init__(self, attn_dropout=0.0, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.attn_dropout = float(attn_dropout)
        self.eps = float(eps)
        self.logit_conv = tf.keras.layers.Conv2D(
            filters=1, kernel_size=(1, 1),
            padding="same", activation=None,
            name="attn_logit_conv"
        )
        self.drop = tf.keras.layers.Dropout(rate=self.attn_dropout, name="attn_weight_dropout")

    def call(self, x, training=None):
        x = tf.convert_to_tensor(x)

        # ---- rank-4: (B,H,W,C) ----
        if x.shape.rank == 4:
            logits = self.logit_conv(x)  # (B,H,W,1)
            B = tf.shape(x)[0]
            H = tf.shape(x)[1]
            W = tf.shape(x)[2]

            logits = tf.reshape(logits, (B, H * W))
            logits = logits - tf.reduce_max(logits, axis=-1, keepdims=True)
            weights = tf.nn.softmax(logits, axis=-1)

            if self.attn_dropout > 0.0:
                weights = self.drop(weights, training=training)
                denom = tf.reduce_sum(weights, axis=-1, keepdims=True)
                weights = weights / tf.maximum(denom, self.eps)

            weights = tf.reshape(weights, (B, H, W, 1))
            pooled  = tf.reduce_sum(x * weights, axis=[1, 2])  # (B,C)
            return pooled

        # ---- rank-5: (B,T,H,W,C) ----
        if x.shape.rank == 5:
            B = tf.shape(x)[0]
            T = tf.shape(x)[1]
            H = tf.shape(x)[2]
            W = tf.shape(x)[3]
            C = tf.shape(x)[4]

            x_bt = tf.reshape(x, (B * T, H, W, C))
            logits = self.logit_conv(x_bt)  # (B*T,H,W,1)

            logits = tf.reshape(logits, (B * T, H * W))
            logits = logits - tf.reduce_max(logits, axis=-1, keepdims=True)
            weights = tf.nn.softmax(logits, axis=-1)

            if self.attn_dropout > 0.0:
                weights = self.drop(weights, training=training)
                denom = tf.reduce_sum(weights, axis=-1, keepdims=True)
                weights = weights / tf.maximum(denom, self.eps)

            weights = tf.reshape(weights, (B * T, H, W, 1))
            pooled  = tf.reduce_sum(x_bt * weights, axis=[1, 2])  # (B*T,C)
            pooled  = tf.reshape(pooled, (B, T, tf.shape(pooled)[-1]))  # (B,T,C)
            return pooled

        raise ValueError(f"SpatialAttentionPooling2D expects rank-4 or rank-5, got rank={x.shape.rank}")

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"attn_dropout": self.attn_dropout, "eps": self.eps})
        return cfg

@tf.keras.utils.register_keras_serializable(package="Custom")
class CoordChannel2D(Layer):
    def __init__(self, h, w, mode="xy_r", use_rbf=False, rings=8, gamma=8.0, **kwargs):
        super().__init__(**kwargs)
        self.h = int(h); self.w = int(w)
        self.mode = str(mode)
        self.use_rbf = bool(use_rbf)
        self.rings = int(rings)
        self.gamma = float(gamma)
        if self.mode == "xy":
            self.cc = 2
        elif self.mode == "xy_r":
            self.cc = 3 + (self.rings if self.use_rbf else 0)
        else:
            raise ValueError("coord mode must be 'xy' or 'xy_r'.")

    def call(self, ref):
        B = tf.shape(ref)[0]
        xs = tf.linspace(-1.0, 1.0, self.w)
        ys = tf.linspace(-1.0, 1.0, self.h)
        xx, yy = tf.meshgrid(xs, ys)

        feats = [xx[..., None], yy[..., None]]
        if self.mode == "xy_r":
            rr = tf.sqrt(tf.maximum(xx * xx + yy * yy, 1e-12))
            rr = tf.clip_by_value(rr / tf.sqrt(2.0), 0.0, 1.0)
            feats.append(rr[..., None])
            if self.use_rbf:
                centers = tf.linspace(0.0, 1.0, self.rings)
                rr3 = rr[..., None]
                c3  = centers[None, None, :]
                rbf = tf.exp(-self.gamma * tf.square(rr3 - c3))
                feats.append(rbf)

        coords = tf.concat(feats, axis=-1)   # (H,W,Cc)
        coords = coords[None, ...]           # (1,H,W,Cc)
        coords = tf.tile(coords, [B, 1, 1, 1])
        return tf.cast(coords, tf.float32)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"h": self.h, "w": self.w, "mode": self.mode, "use_rbf": self.use_rbf, "rings": self.rings, "gamma": self.gamma})
        return cfg

@tf.keras.utils.register_keras_serializable(package="Custom")
class SliceTimeIndex(tf.keras.layers.Layer):
    def __init__(self, index=0, **kwargs):
        super().__init__(**kwargs)
        self.index = int(index)

    def call(self, x):
        return x[:, self.index, ...]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"index": self.index})
        return cfg

@tf.keras.utils.register_keras_serializable(package="Custom")
class SliceTimeRange(tf.keras.layers.Layer):
    def __init__(self, start=0, end=None, **kwargs):
        super().__init__(**kwargs)
        self.start = int(start)
        self.end = None if end is None else int(end)

    def call(self, x):
        return x[:, self.start:self.end, ...]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"start": self.start, "end": self.end})
        return cfg

@tf.keras.utils.register_keras_serializable(package="Custom")
class TileToTime(tf.keras.layers.Layer):
    def __init__(self, T, **kwargs):
        super().__init__(**kwargs)
        self.T = int(T)

    def call(self, x):
        # (B,H,W,C) -> (B,T,H,W,C)
        x = x[:, None, ...]
        return tf.tile(x, [1, self.T, 1, 1, 1])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"T": self.T})
        return cfg

@tf.keras.utils.register_keras_serializable(package="Custom")
class AddLearnedPositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, T_steps, **kwargs):
        super().__init__(**kwargs)
        self.T_steps = int(T_steps)

    def build(self, input_shape):
        C = int(input_shape[-1])
        self.pos_emb = self.add_weight(
            name="pos_emb",
            shape=(self.T_steps, C),
            initializer="zeros",
            trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        # x: (B,T,C)
        return x + self.pos_emb[None, :, :]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"T_steps": self.T_steps})
        return cfg

# ================== 7. Model (REPLACE) ==================
def build_physics_inspired_model(
    branch_input_shapes,
    track_feat_dim,
    T_steps,
    dec_filters=36,
    output_dim=2,
    dropout_rate=0.20,
    adapt_mode="downsample",
    base_hw=81,
    aux_map_hw=21,
    aux_hidden_dim=64,
    use_coords=True,
    coord_mode="xy_r",
    use_rbf_rings=True,
    rbf_rings=8,
    rbf_gamma=8.0,
    per_level_ch=8,
    post_fuse_ch=32,
):
    """
    [NEW]
    - env input sequence = [-6, 0, +6, ..., +H]  => Tin = T_steps + 1
    - output sequence    = [+6, +12, ..., +H]    => length = T_steps - 1
    - target meaning     = cumulative delta from T
    """
    TARGET_H = int(GRID["TARGET_H"])
    TARGET_W = int(GRID["TARGET_W"])

    scale_h = max(1, int(round(TARGET_H / float(base_hw))))
    scale_w = max(1, int(round(TARGET_W / float(base_hw))))
    scale   = max(scale_h, scale_w)

    widths = autoscale_widths_from_dec_filters(dec_filters, scale=scale)
    df             = widths["df"]
    df2            = widths["df2"]
    env_latent_dim = widths["env_latent_dim"]
    fused_channels = widths["fused_channels"]
    attn_heads     = widths["attn_heads"]
    attn_keydim    = widths["attn_keydim"]
    mlp_dim        = widths["mlp_dim"]

    def safe_resize_2d(x4, h_to, w_to, name, force_bilinear=False):
        h = x4.shape[1]
        w = x4.shape[2]
        if (not force_bilinear) and (h is not None) and (w is not None) and (h_to <= h) and (w_to <= w):
            if (h % h_to == 0) and (w % w_to == 0):
                ph = int(h // h_to)
                pw = int(w // w_to)
                return AveragePooling2D(pool_size=(ph, pw), strides=(ph, pw), padding="valid", name=name)(x4)
        return Resizing(h_to, w_to, interpolation="bilinear", name=name)(x4)

    def safe_resize_td(x5, h_to, w_to, name):
        return TimeDistributed(Resizing(h_to, w_to, interpolation="bilinear"), name=name)(x5)

    def tile_static_to_time(x4, T_future, name):
        return TileToTime(T_future, name=name)(x4)

    # ---------- Inputs ----------
    # AUX flat vector: 12 dims
    inp_track = Input(shape=(track_feat_dim,), name="track_init")
    # IR pair: [T-6, T]
    inp_ir = Input(shape=(2, TARGET_H, TARGET_W, 1), name="IR_t0")
    inputs = [inp_track, inp_ir]

    # ---------- ENV branches ----------
    env_inputs, env_state_maps_prev, env_state_maps_curr, env_future_seqs = [], [], [], []

    Tin_ref = None
    for var, shape in branch_input_shapes.items():
        Tin_var, L, Hvar, Wvar = shape
        inp = Input(shape=(Tin_var, L, Hvar, Wvar), name=f"input_{var}")
        env_inputs.append(inp)
        inputs.append(inp)

        if Tin_ref is None:
            Tin_ref = int(Tin_var)
        else:
            if int(Tin_var) != Tin_ref:
                raise ValueError(f"All ENV branches must share same Tin. got {Tin_ref} and {Tin_var}")

        z = Permute((1, 3, 4, 2), name=f"{var}_to_BT_HWC")(inp)  # (Tin,H,W,L)

        grouped_filters = int(L * int(per_level_ch))
        z = TimeDistributed(
            Conv2D(grouped_filters, (3, 3), padding="same", activation=swish, groups=int(L)),
            name=f"{var}_enc_gconv1"
        )(z)
        z = TimeDistributed(
            Conv2D(grouped_filters, (3, 3), padding="same", activation=swish, groups=int(L)),
            name=f"{var}_enc_gconv2"
        )(z)
        z = TimeDistributed(
            Conv2D(int(post_fuse_ch), (1, 1), padding="same", activation=swish),
            name=f"{var}_enc_fuse1x1"
        )(z)

        z_prev   = SliceTimeIndex(index=0, name=f"{var}_prev6")(z)
        z_curr   = SliceTimeIndex(index=1, name=f"{var}_curr")(z)
        z_future = SliceTimeRange(start=2, end=None, name=f"{var}_future")(z)

        env_state_maps_prev.append(z_prev)
        env_state_maps_curr.append(z_curr)
        env_future_seqs.append(z_future)

    if Tin_ref is None:
        raise ValueError("branch_input_shapes is empty.")

    T_future = int(Tin_ref - 2)
    if T_future != int(T_steps - 1):
        raise ValueError(f"T mismatch: env future steps={T_future}, but requested T_steps={T_steps}")

    # concat env encodings
    env_prev = Concatenate(axis=-1, name="env_prev_concat")(env_state_maps_prev)
    env_curr = Concatenate(axis=-1, name="env_curr_concat")(env_state_maps_curr)
    env_fut  = Concatenate(axis=-1, name="env_future_concat")(env_future_seqs)  # (B,T_future,H,W,C)

    if adapt_mode == "downsample" and scale > 1:
        env_prev = safe_resize_2d(env_prev, base_hw, base_hw, name="env_prev_to_base")
        env_curr = safe_resize_2d(env_curr, base_hw, base_hw, name="env_curr_to_base")
        env_fut  = safe_resize_td(env_fut, base_hw, base_hw, name="env_fut_to_base")
        env_h, env_w = base_hw, base_hw
    else:
        env_h = int(env_prev.shape[1]) if env_prev.shape[1] is not None else base_hw
        env_w = int(env_prev.shape[2]) if env_prev.shape[2] is not None else base_hw

    env_prev = Conv2D(env_latent_dim, (1, 1), padding="same", activation=swish, name="env_prev_latent")(env_prev)
    env_curr = Conv2D(env_latent_dim, (1, 1), padding="same", activation=swish, name="env_curr_latent")(env_curr)
    env_fut  = TimeDistributed(
        Conv2D(env_latent_dim, (1, 1), padding="same", activation=swish),
        name="env_future_latent"
    )(env_fut)

    env_delta_pc = tf.keras.layers.Subtract(name="env_curr_minus_prev")([env_curr, env_prev])

    env_curr_tiled = tile_static_to_time(env_curr, T_future, name="env_curr_tile_to_future")
    env_fut_delta = tf.keras.layers.Subtract(name="env_future_minus_curr")([env_fut, env_curr_tiled])

    # ---------- coords ----------
    if use_coords:
        coords = CoordChannel2D(
            env_h,
            env_w,
            mode=coord_mode,
            use_rbf=use_rbf_rings,
            rings=rbf_rings,
            gamma=rbf_gamma,
            name="coords"
        )(inp_track)
    else:
        coords = None

    # ---------- IR pair encoder ----------
    ir_prev = SliceTimeIndex(index=0, name="ir_prev6")(inp_ir)
    ir_curr = SliceTimeIndex(index=1, name="ir_curr")(inp_ir)

    def ir_encoder(x, prefix):
        x = Conv2D(32, (3, 3), padding="same", activation=swish, name=f"{prefix}_conv1")(x)
        x = Conv2D(64, (3, 3), padding="same", activation=swish, name=f"{prefix}_conv2")(x)
        x = Conv2D(96, (3, 3), padding="same", activation=swish, name=f"{prefix}_conv3")(x)
        x = safe_resize_2d(x, env_h, env_w, name=f"{prefix}_to_env_res", force_bilinear=True)
        return x

    ir_prev_f = ir_encoder(ir_prev, "ir_prev")
    ir_curr_f = ir_encoder(ir_curr, "ir_curr")
    ir_delta  = tf.keras.layers.Subtract(name="ir_curr_minus_prev")([ir_curr_f, ir_prev_f])

    # ---------- AUX encoder ----------
    aux_dense = Dense(aux_hidden_dim, activation=swish, name="aux_dense")(inp_track)
    aux_small = Dense(aux_map_hw * aux_map_hw, activation=swish, name="aux_to_map_small")(aux_dense)
    aux_small = Reshape((aux_map_hw, aux_map_hw, 1), name="aux_map_small_reshape")(aux_small)
    aux_map   = safe_resize_2d(aux_small, env_h, env_w, name="aux_to_env_res", force_bilinear=True)
    aux_feat  = Conv2D(24, (3, 3), padding="same", activation=swish, name="aux_feat_conv")(aux_map)

    # ---------- state token ----------
    state_list = [env_prev, env_curr, env_delta_pc, ir_prev_f, ir_curr_f, ir_delta, aux_feat]
    if use_coords:
        state_list.append(coords)

    state_map = Concatenate(axis=-1, name="state_map_concat")(state_list)
    state_map = Conv2D(fused_channels, (3, 3), padding="same", activation=swish, name="state_map_fuse")(state_map)

    # ---------- future forcing sequence ----------
    future_list = [env_fut, env_fut_delta]
    if use_coords:
        coords_seq = tile_static_to_time(coords, T_future, name="coords_future_tile")
        future_list.append(coords_seq)
    aux_seq = tile_static_to_time(aux_feat, T_future, name="aux_future_tile")
    future_list.append(aux_seq)

    future_map = Concatenate(axis=-1, name="future_map_concat")(future_list)
    future_map = TimeDistributed(
        Conv2D(fused_channels, (3, 3), padding="same", activation=swish),
        name="future_map_fuse"
    )(future_map)

    # prepend state map as first token
    state_seq = Reshape((1, env_h, env_w, fused_channels), name="state_map_to_seq")(state_map)
    x_seq = Concatenate(axis=1, name="convlstm_input_seq")([state_seq, future_map])  # len = T_steps

    # ---------- ConvLSTM decoder ----------
    x = ConvLSTM2D(
        df, (3, 3),
        padding="same",
        return_sequences=True,
        activation=swish,
        name="dec_lstm_1"
    )(x_seq)

    x = ConvLSTM2D(
        df2, (3, 3),
        padding="same",
        return_sequences=True,
        activation=swish,
        name="dec_lstm_2"
    )(x)

    x = Dropout(dropout_rate, name="drop_after_lstm")(x)

    # state token 제거 => future horizons만 남김
    x = SliceTimeRange(start=1, end=None, name="drop_state_token")(x)  # (B,T_future,H,W,C)

    x = SpatialAttentionPooling2D(attn_dropout=0.0, name="spatial_attn_pool")(x)  # (B,T_future,C)
    x = AddLearnedPositionalEmbedding(T_future, name="t_pos_emb")(x)

    attn = MultiHeadAttention(
        num_heads=attn_heads,
        key_dim=attn_keydim,
        dropout=dropout_rate,
        name="t_mha"
    )
    y = attn(x, x, use_causal_mask=True)
    x = Add(name="t_attn_residual")([x, y])
    x = LayerNormalization(name="t_attn_ln")(x)

    C_dec = int(x.shape[-1]) if x.shape[-1] is not None else df2

    ff = Dense(mlp_dim, activation=swish, name="t_ffn_1")(x)
    ff = Dropout(dropout_rate, name="t_ffn_drop")(ff)
    ff = Dense(C_dec, activation=None, name="t_ffn_2")(ff)
    x  = Add(name="t_ffn_residual")([x, ff])
    x  = LayerNormalization(name="t_ffn_ln")(x)

    # ---------- Heads ----------
    h_shared = Dense(96, activation=swish, name="mlp_shared_1")(x)
    h_shared = Dropout(dropout_rate, name="mlp_shared_dropout")(h_shared)

    h_mws = Dense(64, activation=swish, name="mlp_mws_1")(h_shared)
    h_mws = Dropout(dropout_rate, name="mlp_mws_drop")(h_mws)
    out_mws = Dense(1, activation=None, name="head_mws")(h_mws)

    h_mslp = Dense(64, activation=swish, name="mlp_mslp_1")(h_shared)
    h_mslp = Dropout(dropout_rate, name="mlp_mslp_drop")(h_mslp)
    out_mslp = Dense(1, activation=None, name="head_mslp")(h_mslp)

    out = Concatenate(axis=-1, name="cum_delta_all_leads")([out_mws, out_mslp])  # (B,T_future,2)

    model = Model(
        inputs=[inp_track, inp_ir] + env_inputs,
        outputs=out,
        name=f"Physics_IR_CUMDELTA_statePrevCurr_futureENV_{adapt_mode}_df{dec_filters}_H{TARGET_H}W{TARGET_W}"
    )
    return model

# ================== 8. Train/Val/Test ==================
def run_train_val_test_multibranch(
    train_years, val_years, test_years,
    all_years, local_env_dir, MAX_FORECAST_HOURS, save_root,
    dec_filters=36, dropout_rate=0.2, learning_rate=2e-4
):
    """
    [UPDATED POLICY]
    - AMP는 MWS에만 적용
    - MSLP는 target/output은 유지하지만 AMP 없이 일반 masked Huber만 적용
    - AMP는 +6h ~ +24h까지만 적용

    [PLOT FIX]
    - pred_records는 cum_mask_te로 자르지 않음
    - 각 t0마다 0~48h 전체 예측을 저장
    - truth는 관측 없으면 NaN으로 저장
    - 평가(R2/RMSE/RI-RW)는 기존처럼 cum_mask_te 기준 유지
    """
    global _PREPROC_CACHE

    AMP_CONST    = 2.0
    AMP_CLIP_MIN = 1.0
    AMP_CLIP_MAX = AMP_CONST
    AMP_MAX_HOUR = 24.0

    AMP_AVG6_ON   = 6.0
    AMP_AVG6_FULL = 18.0

    print(f"\n################### Train={train_years}, Val={val_years}, Test={test_years} ###################")
    print(f"### Hyperparams: dec_filters={dec_filters}, dropout_rate={dropout_rate}, lr={learning_rate} ###")
    print("### UPDATED POLICY: partial rollout allowed | valid leads only | AMP uses MWS only ###")
    print(f"### AMP ACTIVE RANGE: +6h ~ +{int(AMP_MAX_HOUR)}h only ###")
    print("### PLOT FIX: pred_records stores full 0~48h predictions without cum_mask trimming ###")

    T = MAX_FORECAST_HOURS // 6 + 1
    Tin = T + 1
    lead_hours = list(range(0, MAX_FORECAST_HOURS + 1, 6))
    fold_seed = int(SEED + int(test_years[0]) if len(test_years) > 0 else SEED)

    cache_key = (
        tuple(train_years),
        tuple(val_years),
        tuple(test_years),
        int(MAX_FORECAST_HOURS),
        str(GRID.get("RES_MODE", "")),
        bool(USE_IR),
        "AMP_MWS_ONLY_UPTO24H_V1",
        float(AMP_AVG6_ON),
        float(AMP_AVG6_FULL),
        float(AMP_CONST),
        float(AMP_MAX_HOUR),
    )

    def make_ds_dict(input_names, inputs_list, y=None, batch_size=48, shuffle=True, seed=123, drop_remainder=True):
        assert len(input_names) == len(inputs_list), (len(input_names), len(inputs_list))

        x_dict = {name: arr for name, arr in zip(input_names, inputs_list)}
        x_ds = tf.data.Dataset.from_tensor_slices(x_dict)

        if y is None:
            ds = x_ds
            n = int(inputs_list[0].shape[0])
        else:
            y_ds = tf.data.Dataset.from_tensor_slices(y)
            ds = tf.data.Dataset.zip((x_ds, y_ds))
            n = int(y.shape[0])

        if shuffle:
            ds = ds.shuffle(buffer_size=n, seed=int(seed), reshuffle_each_iteration=True)

        ds = ds.batch(int(batch_size), drop_remainder=bool(drop_remainder))
        ds = ds.prefetch(tf.data.AUTOTUNE)
        return ds

    def make_cumulative_delta_masked(y_obs, mask_abs):
        y_obs    = np.asarray(y_obs, dtype=np.float32)
        mask_abs = np.asarray(mask_abs, dtype=np.float32)

        B_, T_ = y_obs.shape
        delta = np.zeros((B_, T_), dtype=np.float32)

        for t in range(1, T_):
            valid = (
                (mask_abs[:, 0] > 0.5) &
                (mask_abs[:, t] > 0.5) &
                np.isfinite(y_obs[:, 0]) &
                np.isfinite(y_obs[:, t])
            )
            if np.any(valid):
                delta[valid, t] = (y_obs[valid, t] - y_obs[valid, 0]).astype(np.float32)

        delta[:, 0] = 0.0
        delta[~np.isfinite(delta)] = 0.0
        return delta

    def make_cumulative_valid_mask(mask_abs, env_valid_mask):
        mask_abs = np.asarray(mask_abs, dtype=np.float32)
        env_valid_mask = np.asarray(env_valid_mask, dtype=np.float32)

        B_, T_ = mask_abs.shape
        cum_mask = np.zeros((B_, T_), dtype=np.float32)
        cum_mask[:, 0] = 1.0

        for t in range(1, T_):
            cum_mask[:, t] = (
                (mask_abs[:, 0] > 0.5) &
                (mask_abs[:, t] > 0.5) &
                (env_valid_mask[:, t] > 0.5)
            ).astype(np.float32)

        cum_mask[~np.isfinite(cum_mask)] = 0.0
        return cum_mask

    def fit_mu_std_per_lead_masked(y_delta, valid_mask, eps=1e-6):
        y_delta    = np.asarray(y_delta, dtype=np.float32)
        valid_mask = np.asarray(valid_mask, dtype=np.float32)

        B_, T_ = y_delta.shape
        mu = np.zeros((T_,), dtype=np.float32)
        sd = np.ones((T_,), dtype=np.float32)

        mu[0] = 0.0
        sd[0] = 1.0

        for t in range(1, T_):
            valid = (valid_mask[:, t] > 0.5) & np.isfinite(y_delta[:, t])
            vals = y_delta[valid, t]
            if vals.size == 0:
                mu[t], sd[t] = 0.0, 1.0
            else:
                mt = float(np.mean(vals))
                st = float(np.std(vals))
                if not np.isfinite(mt):
                    mt = 0.0
                if (not np.isfinite(st)) or st < eps:
                    st = 1.0
                mu[t], sd[t] = mt, st
        return mu, sd

    def zscore_per_lead_masked(y_delta, valid_mask, mu_vec, sd_vec, eps=1e-6):
        y_delta    = np.asarray(y_delta, dtype=np.float32)
        valid_mask = np.asarray(valid_mask, dtype=np.float32)
        mu_vec     = np.asarray(mu_vec, dtype=np.float32)
        sd_vec     = np.asarray(sd_vec, dtype=np.float32)

        z = np.zeros_like(y_delta, dtype=np.float32)
        z[:, 0] = 0.0

        for t in range(1, y_delta.shape[1]):
            valid = (valid_mask[:, t] > 0.5) & np.isfinite(y_delta[:, t])
            if np.any(valid):
                z[valid, t] = ((y_delta[valid, t] - mu_vec[t]) / (sd_vec[t] + eps)).astype(np.float32)

        z[~np.isfinite(z)] = 0.0
        return z

    def build_amp_from_avg6_mws_only(
        y_cum_mws,
        final_valid_mask,
        lead_hours,
        amp_const=2.0,
        th_on=7.5,
        th_full=15.0,
        amp_max_hour=24.0,
    ):
        y_cum_mws        = np.asarray(y_cum_mws, dtype=np.float32)
        final_valid_mask = np.asarray(final_valid_mask, dtype=np.float32)

        B_, T_ = y_cum_mws.shape
        if len(lead_hours) != T_:
            raise ValueError(f"lead_hours 길이({len(lead_hours)})와 target dim({T_})이 다릅니다.")

        amp_mws = np.ones((B_, T_), dtype=np.float32)
        denom = max(float(th_full - th_on), 1e-6)

        for t in range(1, T_):
            h = float(lead_hours[t])

            if h > float(amp_max_hour):
                continue

            n6 = h / 6.0
            if n6 <= 0:
                continue

            valid = (
                (final_valid_mask[:, t] > 0.5) &
                np.isfinite(y_cum_mws[:, t])
            )

            if not np.any(valid):
                continue

            avg_mws = y_cum_mws[valid, t] / n6
            mag_mws = np.abs(avg_mws).astype(np.float32)

            strength = np.clip((mag_mws - float(th_on)) / denom, 0.0, 1.0).astype(np.float32)
            amp_mws[valid, t] = 1.0 + (float(amp_const) - 1.0) * strength

        amp_mws[:, 0] = 1.0
        amp_mws = np.clip(amp_mws, AMP_CLIP_MIN, AMP_CLIP_MAX).astype(np.float32)
        amp_mws[~np.isfinite(amp_mws)] = 1.0
        return amp_mws.astype(np.float32)

    def pack_y(z_mws, z_mslp, valid_mask, amp_mws):
        z_mws      = np.asarray(z_mws,      dtype=np.float32)
        z_mslp     = np.asarray(z_mslp,     dtype=np.float32)
        valid_mask = np.asarray(valid_mask, dtype=np.float32)
        amp_mws    = np.asarray(amp_mws,    dtype=np.float32)

        B_, T_ = z_mws.shape
        y = np.zeros((B_, T_, 5), dtype=np.float32)
        y[:, :, 0] = z_mws
        y[:, :, 1] = z_mslp
        y[:, :, 2] = valid_mask
        y[:, :, 3] = amp_mws
        y[:, :, 4] = 1.0

        y[:, 0, 0:2] = 0.0
        y[:, 0, 2]   = 0.0
        y[:, 0, 3]   = 1.0
        y[:, 0, 4]   = 1.0

        miss = (valid_mask <= 0.5)
        y[:, :, 0][miss] = 0.0
        y[:, :, 1][miss] = 0.0
        y[~np.isfinite(y)] = 0.0
        return y

    def fit_aux_scaler_generic(X_aux_train, eps=1e-6):
        X = np.asarray(X_aux_train, dtype=np.float32)
        if X.ndim != 2:
            raise ValueError(f"X_aux_train must be 2D. got {X.shape}")

        mu = np.zeros((X.shape[1],), dtype=np.float32)
        sd = np.ones((X.shape[1],), dtype=np.float32)

        keep_raw_index = int(X.shape[1] - 1)

        for i in range(X.shape[1]):
            if i == keep_raw_index:
                mu[i], sd[i] = 0.0, 1.0
                continue

            vals = X[:, i]
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                mu[i], sd[i] = 0.0, 1.0
            else:
                mu[i] = float(np.mean(vals))
                s = float(np.std(vals))
                sd[i] = 1.0 if ((not np.isfinite(s)) or s < eps) else s

        return {
            "mu": mu.astype(np.float32),
            "std": sd.astype(np.float32),
            "keep_raw_index": keep_raw_index
        }

    def apply_aux_scaler_generic(X_aux, scaler, eps=1e-6):
        X = np.asarray(X_aux, dtype=np.float32)
        mu = np.asarray(scaler["mu"], dtype=np.float32)
        sd = np.asarray(scaler["std"], dtype=np.float32)
        keep_raw_index = int(scaler["keep_raw_index"])

        Z = X.copy()
        for i in range(X.shape[1]):
            if i == keep_raw_index:
                Z[:, i] = X[:, i]
            else:
                Z[:, i] = (X[:, i] - mu[i]) / (sd[i] + eps)

        Z[~np.isfinite(Z)] = 0.0
        return Z.astype(np.float32)

    if cache_key in _PREPROC_CACHE:
        print("[PREPROC CACHE] HIT -> reuse split-level preprocessed data")
        cache = _PREPROC_CACHE[cache_key]

    else:
        print("[PREPROC CACHE] MISS -> build split-level preprocessed data")

        df_train = load_filtered_df_years(TRACK_DIR, train_years)
        df_val   = load_filtered_df_years(TRACK_DIR, val_years)
        df_test  = load_filtered_df_years(TRACK_DIR, test_years)

        (
            Xtr_env, Xtr_aux, Xtr_ir,
            ytr_obs_mws, ytr_obs_mslp,
            ytr_ff_mws, ytr_ff_mslp,
            mask_tr, env_mask_tr, tr_meta
        ) = create_samples(df_train, "train", local_env_dir, MAX_FORECAST_HOURS)

        (
            Xv_env, Xv_aux, Xv_ir,
            yv_obs_mws, yv_obs_mslp,
            yv_ff_mws, yv_ff_mslp,
            mask_v, env_mask_v, v_meta
        ) = create_samples(df_val, "val", local_env_dir, MAX_FORECAST_HOURS)

        (
            Xte_env, Xte_aux, Xte_ir,
            yte_obs_mws, yte_obs_mslp,
            yte_ff_mws, yte_ff_mslp,
            mask_te, env_mask_te, te_meta
        ) = create_samples(df_test, "test", local_env_dir, MAX_FORECAST_HOURS)

        gc.collect()
        print(f"[Samples Created] Train={ytr_obs_mws.shape[0]}, Val={yv_obs_mws.shape[0]}, Test={yte_obs_mws.shape[0]}")

        if (ytr_obs_mws.shape[0] == 0) or (yv_obs_mws.shape[0] == 0) or (yte_obs_mws.shape[0] == 0):
            print("[ERR] One of train/val/test has zero samples.")
            return (
                None, df_test, "TEST", None,
                None,
                None, None,
                None, None,
                None, [], te_meta, None
            )

        Xtr_env, Xtr_ir, scalers = normalize_features(Xtr_env, Xtr_ir, fit=True)
        Xv_env,  Xv_ir,  _       = normalize_features(Xv_env,  Xv_ir,  scalers=scalers, fit=False)
        Xte_env, Xte_ir, _       = normalize_features(Xte_env, Xte_ir, scalers=scalers, fit=False)

        aux_scaler = fit_aux_scaler_generic(Xtr_aux)
        Xtr_aux = apply_aux_scaler_generic(Xtr_aux, aux_scaler)
        Xv_aux  = apply_aux_scaler_generic(Xv_aux,  aux_scaler)
        Xte_aux = apply_aux_scaler_generic(Xte_aux, aux_scaler)

        debug_plot_all_first_normalized_fields_save_and_show_T0(
            Xte_env, Xte_ir[:, 1], save_root, meta_list=te_meta
        )

        cum_mask_tr = make_cumulative_valid_mask(mask_tr, env_mask_tr)
        cum_mask_v  = make_cumulative_valid_mask(mask_v,  env_mask_v)
        cum_mask_te = make_cumulative_valid_mask(mask_te, env_mask_te)

        ytr_cum_mws  = make_cumulative_delta_masked(ytr_obs_mws,  mask_tr)
        ytr_cum_mslp = make_cumulative_delta_masked(ytr_obs_mslp, mask_tr)
        yv_cum_mws   = make_cumulative_delta_masked(yv_obs_mws,   mask_v)
        yv_cum_mslp  = make_cumulative_delta_masked(yv_obs_mslp,  mask_v)
        yte_cum_mws  = make_cumulative_delta_masked(yte_obs_mws,  mask_te)
        yte_cum_mslp = make_cumulative_delta_masked(yte_obs_mslp, mask_te)

        mu_mws_vec,  std_mws_vec  = fit_mu_std_per_lead_masked(ytr_cum_mws,  cum_mask_tr)
        mu_mslp_vec, std_mslp_vec = fit_mu_std_per_lead_masked(ytr_cum_mslp, cum_mask_tr)

        ztr_mws  = zscore_per_lead_masked(ytr_cum_mws,  cum_mask_tr, mu_mws_vec,  std_mws_vec)
        ztr_mslp = zscore_per_lead_masked(ytr_cum_mslp, cum_mask_tr, mu_mslp_vec, std_mslp_vec)
        zv_mws   = zscore_per_lead_masked(yv_cum_mws,   cum_mask_v,  mu_mws_vec,  std_mws_vec)
        zv_mslp  = zscore_per_lead_masked(yv_cum_mslp,  cum_mask_v,  mu_mslp_vec, std_mslp_vec)
        zte_mws  = zscore_per_lead_masked(yte_cum_mws,  cum_mask_te, mu_mws_vec,  std_mws_vec)
        zte_mslp = zscore_per_lead_masked(yte_cum_mslp, cum_mask_te, mu_mslp_vec, std_mslp_vec)

        amp_tr_mws = build_amp_from_avg6_mws_only(
            ytr_cum_mws, cum_mask_tr, lead_hours,
            amp_const=AMP_CONST,
            th_on=AMP_AVG6_ON,
            th_full=AMP_AVG6_FULL,
            amp_max_hour=AMP_MAX_HOUR,
        )
        amp_v_mws = build_amp_from_avg6_mws_only(
            yv_cum_mws, cum_mask_v, lead_hours,
            amp_const=AMP_CONST,
            th_on=AMP_AVG6_ON,
            th_full=AMP_AVG6_FULL,
            amp_max_hour=AMP_MAX_HOUR,
        )
        amp_te_mws = build_amp_from_avg6_mws_only(
            yte_cum_mws, cum_mask_te, lead_hours,
            amp_const=AMP_CONST,
            th_on=AMP_AVG6_ON,
            th_full=AMP_AVG6_FULL,
            amp_max_hour=AMP_MAX_HOUR,
        )

        ytr_all = pack_y(ztr_mws, ztr_mslp, cum_mask_tr, amp_tr_mws)
        yv_all  = pack_y(zv_mws,  zv_mslp,  cum_mask_v,  amp_v_mws)
        yte_all = pack_y(zte_mws, zte_mslp, cum_mask_te, amp_te_mws)

        env_key_order = [k for k in ["ERH", "EDIVE", "EVORT", "EVWS", "ETHETA", "SST", "TOPO"] if k in Xtr_env]

        for k in env_key_order:
            Xtr_env[k] = Xtr_env[k].astype(np.float16, copy=False)
            Xv_env[k]  = Xv_env[k].astype(np.float16, copy=False)
            Xte_env[k] = Xte_env[k].astype(np.float16, copy=False)

        Xtr_ir = Xtr_ir.astype(np.float16, copy=False)
        Xv_ir  = Xv_ir.astype(np.float16, copy=False)
        Xte_ir = Xte_ir.astype(np.float16, copy=False)

        Xtr_aux = Xtr_aux.astype(np.float32, copy=False)
        Xv_aux  = Xv_aux.astype(np.float32, copy=False)
        Xte_aux = Xte_aux.astype(np.float32, copy=False)

        ytr_all = ytr_all.astype(np.float32, copy=False)
        yv_all  = yv_all.astype(np.float32, copy=False)
        yte_all = yte_all.astype(np.float32, copy=False)

        abs_scalers = {
            "mws": {
                "mean": mu_mws_vec.astype(np.float32),
                "std": std_mws_vec.astype(np.float32),
                "mode": "cumulative_delta_per_lead_masked"
            },
            "mslp": {
                "mean": mu_mslp_vec.astype(np.float32),
                "std": std_mslp_vec.astype(np.float32),
                "mode": "cumulative_delta_per_lead_masked"
            },
            "target_mode": "cum_delta_from_T_per_lead_zscore(train_observed_masked)",
            "reconstruction_mode": "absolute = I(T) + cumulative_delta(T->h)",
            "amp_mode": "mws_only_upto24h",
            "amp_cfg": {
                "amp_const": float(AMP_CONST),
                "threshold_on": float(AMP_AVG6_ON),
                "threshold_full": float(AMP_AVG6_FULL),
                "amp_max_hour": float(AMP_MAX_HOUR),
                "definition": "abs( cumulative_change / (lead_hour/6) )",
                "mws_amp_rule": "bidirectional_absolute_average_change",
                "mslp_amp_rule": "not_applied",
            }
        }

        cache = dict(
            df_test=df_test,

            Xtr_env=Xtr_env, Xv_env=Xv_env, Xte_env=Xte_env,
            Xtr_aux=Xtr_aux, Xv_aux=Xv_aux, Xte_aux=Xte_aux,
            Xtr_ir=Xtr_ir,   Xv_ir=Xv_ir,   Xte_ir=Xte_ir,

            tr_meta=tr_meta, v_meta=v_meta, te_meta=te_meta,

            ytr_all=ytr_all, yv_all=yv_all, yte_all=yte_all,

            yv_obs_mws=yv_obs_mws, yv_obs_mslp=yv_obs_mslp,
            yte_obs_mws=yte_obs_mws, yte_obs_mslp=yte_obs_mslp,

            yte_ff_mws=yte_ff_mws, yte_ff_mslp=yte_ff_mslp,

            t0_v_mws=yv_obs_mws[:, 0],
            t0_v_mslp=yv_obs_mslp[:, 0],
            t0_te_mws=yte_obs_mws[:, 0],
            t0_te_mslp=yte_obs_mslp[:, 0],

            mask_v=mask_v,
            mask_te=mask_te,
            env_mask_tr=env_mask_tr,
            env_mask_v=env_mask_v,
            env_mask_te=env_mask_te,
            cum_mask_tr=cum_mask_tr,
            cum_mask_v=cum_mask_v,
            cum_mask_te=cum_mask_te,

            T=T,
            Tin=Tin,
            env_key_order=env_key_order,
            abs_scalers=abs_scalers,
            fold_seed=fold_seed,
        )

        _PREPROC_CACHE[cache_key] = cache
        print("[PREPROC CACHE] Stored split-level preprocessed data")

    df_test = cache["df_test"]

    Xtr_env, Xv_env, Xte_env = cache["Xtr_env"], cache["Xv_env"], cache["Xte_env"]
    Xtr_aux, Xv_aux, Xte_aux = cache["Xtr_aux"], cache["Xv_aux"], cache["Xte_aux"]
    Xtr_ir,  Xv_ir,  Xte_ir  = cache["Xtr_ir"],  cache["Xv_ir"],  cache["Xte_ir"]

    ytr_all, yv_all, yte_all = cache["ytr_all"], cache["yv_all"], cache["yte_all"]

    y_true_test_obs_mws  = cache["yte_obs_mws"]
    y_true_test_obs_mslp = cache["yte_obs_mslp"]
    y_true_test_ff_mws   = cache["yte_ff_mws"]
    y_true_test_ff_mslp  = cache["yte_ff_mslp"]

    t0_te_mws   = cache["t0_te_mws"]
    t0_te_mslp  = cache["t0_te_mslp"]
    cum_mask_te = cache["cum_mask_te"]

    T             = cache["T"]
    Tin           = cache["Tin"]
    env_key_order = cache["env_key_order"]
    abs_scalers   = cache["abs_scalers"]
    te_meta       = cache["te_meta"]
    fold_seed     = int(cache.get("fold_seed", fold_seed))

    track_feat_dim = int(Xtr_aux.shape[1])
    print(f"[AUX DIM] track_feat_dim={track_feat_dim}")
    print(f"[ENV ORDER] {env_key_order}")
    print(f"[ENV Tin] {Tin} = [-6, 0, +6, ..., +H]")

    from collections import OrderedDict
    inp_shapes = OrderedDict((k, Xtr_env[k].shape[1:]) for k in env_key_order)

    cleanup_tf()

    model_multi = build_physics_inspired_model(
        inp_shapes,
        track_feat_dim=track_feat_dim,
        T_steps=T,
        dec_filters=dec_filters,
        dropout_rate=dropout_rate,
        output_dim=2,
        adapt_mode="downsample",
        base_hw=81,
        use_coords=True,
        coord_mode="xy_r",
        use_rbf_rings=True,
        rbf_rings=8,
        rbf_gamma=8.0,
        per_level_ch=8,
        post_fuse_ch=32,
    )

    optimizer = AdamW(learning_rate=learning_rate, weight_decay=5e-4)
    model_multi.compile(optimizer=optimizer, loss=physical_loss)

    input_names = ["track_init", "IR_t0"] + [f"input_{k}" for k in env_key_order]

    train_inputs = [Xtr_aux, Xtr_ir] + [Xtr_env[k] for k in env_key_order]
    val_inputs   = [Xv_aux,  Xv_ir ] + [Xv_env[k]  for k in env_key_order]
    test_inputs  = [Xte_aux, Xte_ir] + [Xte_env[k] for k in env_key_order]

    train_ds = make_ds_dict(
        input_names, train_inputs,
        y=ytr_all, batch_size=48,
        shuffle=True, seed=fold_seed,
        drop_remainder=True
    )

    val_ds = make_ds_dict(
        input_names, val_inputs,
        y=yv_all, batch_size=48,
        shuffle=False, seed=fold_seed,
        drop_remainder=False
    )

    test_ds = make_ds_dict(
        input_names, test_inputs,
        y=None, batch_size=48,
        shuffle=False, seed=fold_seed,
        drop_remainder=False
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=8,
        restore_best_weights=True
    )

    lr_sched = ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=2
    )

    model_multi.fit(
        train_ds,
        validation_data=val_ds,
        epochs=100,
        callbacks=[early_stop, lr_sched],
        verbose=2
    )

    model_path = os.path.join(save_root, "model_physics_inspired_IR.keras")
    try:
        model_multi.save(model_path)
        print(f"✅ Model saved: {model_path}")
    except Exception as e:
        print(f"[WARN] model save failed: {e}")

    pred = model_multi.predict(test_ds, verbose=2)

    mu_mws_vec   = np.asarray(abs_scalers["mws"]["mean"],  dtype=np.float32)
    std_mws_vec  = np.asarray(abs_scalers["mws"]["std"],   dtype=np.float32)
    mu_mslp_vec  = np.asarray(abs_scalers["mslp"]["mean"], dtype=np.float32)
    std_mslp_vec = np.asarray(abs_scalers["mslp"]["std"],  dtype=np.float32)

    pred_cum_mws_leads  = pred[:, :, 0].astype(np.float32) * std_mws_vec[None, 1:]  + mu_mws_vec[None, 1:]
    pred_cum_mslp_leads = pred[:, :, 1].astype(np.float32) * std_mslp_vec[None, 1:] + mu_mslp_vec[None, 1:]

    zero0 = np.zeros((pred.shape[0], 1), dtype=np.float32)
    pred_cum_mws  = np.concatenate([zero0, pred_cum_mws_leads], axis=1)
    pred_cum_mslp = np.concatenate([zero0, pred_cum_mslp_leads], axis=1)

    pred_mws  = t0_te_mws[:, None].astype(np.float32)  + pred_cum_mws
    pred_mslp = t0_te_mslp[:, None].astype(np.float32) + pred_cum_mslp

    # =========================
    # ✅ FIXED: pred_records는 cum_mask_te로 자르지 않는다
    # =========================
    pred_records = defaultdict(list)
    for i, meta in enumerate(te_meta):
        t0 = pd.to_datetime(meta["t0"])
        tid = f"{meta['basin']}{str(meta['id']).zfill(2)}"

        for j, h in enumerate(lead_hours):
            yt_mws = float(y_true_test_obs_mws[i, j]) if np.isfinite(y_true_test_obs_mws[i, j]) else np.nan
            yp_mws = float(pred_mws[i, j]) if np.isfinite(pred_mws[i, j]) else np.nan

            yt_mslp = float(y_true_test_obs_mslp[i, j]) if np.isfinite(y_true_test_obs_mslp[i, j]) else np.nan
            yp_mslp = float(pred_mslp[i, j]) if np.isfinite(pred_mslp[i, j]) else np.nan

            pred_records[tid].append([t0, int(h), "MWS", yt_mws, yp_mws])
            pred_records[tid].append([t0, int(h), "MSLP", yt_mslp, yp_mslp])

    # =========================
    # 평가지표는 기존대로 cum_mask_te 사용
    # =========================
    overall_r2_records = []
    print("\n📈 [R² per lead time | obs-valid AND env-valid]")
    for j, hour in enumerate(lead_hours):
        valid_mws  = (cum_mask_te[:, j] > 0.5) & np.isfinite(y_true_test_obs_mws[:, j])  & np.isfinite(pred_mws[:, j])
        valid_mslp = (cum_mask_te[:, j] > 0.5) & np.isfinite(y_true_test_obs_mslp[:, j]) & np.isfinite(pred_mslp[:, j])

        r2_mws  = r2_score(y_true_test_obs_mws[valid_mws,  j], pred_mws[valid_mws,  j])  if np.sum(valid_mws)  >= 3 else np.nan
        r2_mslp = r2_score(y_true_test_obs_mslp[valid_mslp, j], pred_mslp[valid_mslp, j]) if np.sum(valid_mslp) >= 3 else np.nan

        print(f"▶ Lead Time +{hour:2}h | MWS R²: {r2_mws:.4f} | MSLP R²: {r2_mslp:.4f}")
        overall_r2_records.append([hour, r2_mws, r2_mslp])

    df_overall_r2 = pd.DataFrame(overall_r2_records, columns=["LeadHour", "R2_MWS", "R2_MSLP"])
    r2_path = os.path.join(save_root, "r2_overall.csv")
    df_overall_r2.to_csv(r2_path, index=False)
    print(f"✅ Overall R² saved: {r2_path}")

    overall_rmse_records = []
    print("\n📉 [RMSE per lead time | obs-valid AND env-valid]")
    for j, hour in enumerate(lead_hours):
        valid_mws  = (cum_mask_te[:, j] > 0.5) & np.isfinite(y_true_test_obs_mws[:, j])  & np.isfinite(pred_mws[:, j])
        valid_mslp = (cum_mask_te[:, j] > 0.5) & np.isfinite(y_true_test_obs_mslp[:, j]) & np.isfinite(pred_mslp[:, j])

        rmse_mws  = np.sqrt(mean_squared_error(y_true_test_obs_mws[valid_mws,  j], pred_mws[valid_mws,  j]))  if np.sum(valid_mws)  > 0 else np.nan
        rmse_mslp = np.sqrt(mean_squared_error(y_true_test_obs_mslp[valid_mslp, j], pred_mslp[valid_mslp, j])) if np.sum(valid_mslp) > 0 else np.nan

        print(f"▶ Lead Time +{hour:2}h | MWS RMSE: {rmse_mws:.4f} | MSLP RMSE: {rmse_mslp:.4f}")
        overall_rmse_records.append([hour, rmse_mws, rmse_mslp])

    df_overall_rmse = pd.DataFrame(overall_rmse_records, columns=["LeadHour", "RMSE_MWS", "RMSE_MSLP"])
    overall_path = os.path.join(save_root, "rmse_overall.csv")
    df_overall_rmse.to_csv(overall_path, index=False)
    print(f"✅ Overall RMSE saved: {overall_path}")

    horizon_hours = 24
    TS_MIN_KT = 34.0
    TS_ON = "both"

    try:
        ri_stats, rw_stats = compute_ri_rw_contingency(
            y_true_test_obs_mws, pred_mws, cum_mask_te, lead_hours,
            ri_thresh=30.0, rw_thresh=30.0,
            horizon_hours=horizon_hours, min_intensity_kt=TS_MIN_KT, ts_on=TS_ON
        )

        lines = []
        for d in [ri_stats, rw_stats]:
            lines.append(f"[{d['event']}] horizon={d['horizon_hours']}h, threshold={d['threshold_kt']} kt")
            lines.append(f"  TS filter: min_intensity_kt={d.get('min_intensity_kt', None)}, ts_on={d.get('ts_on', None)}")
            lines.append(f"  H={d['H']}  M={d['M']}  F={d['F']}  CN={d['CN']}")
            lines.append(f"  POD={d['POD']:.3f} FAR={d['FAR']:.3f} CSI={d['CSI']:.3f} BIAS={d['BIAS']:.3f}")
            lines.append("")

        txt_path = os.path.join(save_root, f"contingency_RI_RW_{horizon_hours}h.txt")
        with open(txt_path, "w") as f:
            f.write("\n".join(lines))
        print(f"✅ RI/RW {horizon_hours}h contingency saved (txt): {txt_path}")

        save_ri_rw_detailed_excel(
            y_true_mws=y_true_test_obs_mws, y_pred_mws=pred_mws, mask=cum_mask_te,
            lead_hours=lead_hours, meta_list=te_meta,
            ri_stats=ri_stats, rw_stats=rw_stats,
            horizon_hours=horizon_hours, save_root=save_root,
            ri_thresh=30.0, rw_thresh=30.0,
            min_intensity_kt=TS_MIN_KT, ts_on=TS_ON
        )
    except Exception as e:
        print(f"[WARN] RI/RW contingency computation failed: {e}")

    test_tag = "TEST"
    idx_te = list(range(y_true_test_obs_mws.shape[0]))

    try:
        del train_ds, val_ds, test_ds
    except Exception:
        pass
    gc.collect()

    return (
        pred_records, df_test, test_tag, T,
        env_key_order,
        y_true_test_ff_mws, y_true_test_ff_mslp,
        y_true_test_obs_mws[:, 0], y_true_test_obs_mslp[:, 0],
        abs_scalers, idx_te, te_meta, cum_mask_te
    )

# ================== 9. Visualization ==================
def save_legend(output_path, labels, colors=None, markers=None):
    handles = []
    for i, label in enumerate(labels):
        color = colors[i] if colors else "black"
        marker = markers[i] if markers else "o"
        line = mlines.Line2D(
            [], [], color=color, marker=marker,
            linestyle="-" if label == "True" else "--",
            label=label, markersize=8
        )
        handles.append(line)
    fig, ax = plt.subplots()
    ax.legend(handles=handles, loc="center", frameon=True)
    ax.axis("off")
    fig.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight", transparent=True)
    plt.close()
    print(f"🎯 범례 저장 완료: {output_path}")

def save_rollouts_to_excel(pred_records, save_root, storm_id=None, max_forecast_hours=48, df_test=None):
    os.makedirs(save_root, exist_ok=True)

    rows = []
    for sid, recs in pred_records.items():
        for r in recs:
            t0, h, target, y_true, y_pred = r
            t0 = pd.to_datetime(t0)
            h = int(h)
            if h < 0 or h > max_forecast_hours:
                continue
            if not np.isfinite(y_pred):
                continue
            vt = t0 + pd.Timedelta(hours=h)
            rows.append([sid, t0, h, vt, str(target), float(y_pred)])

    if len(rows) == 0:
        print("[WARN] pred_records가 비어 있어 rollout 엑셀 저장을 건너뜁니다.")
        return None

    df_pred_raw = pd.DataFrame(rows, columns=["StormID", "t0", "lead_hour", "valid_time", "Target", "Pred"]).sort_values(
        ["StormID", "t0", "Target", "lead_hour"]
    )

    true_rows = []
    if df_test is not None and len(df_test) > 0:
        for (basin, tid), df_tc in df_test.groupby(["basin", "id"]):
            sid = f"{basin}{str(tid).zfill(2)}"
            df_tc = df_tc.sort_values("datetime")
            for _, rr in df_tc.iterrows():
                true_rows.append([sid, pd.to_datetime(rr["datetime"]), float(rr["mslp"]), float(rr["mws"])])

    df_true_track = pd.DataFrame(true_rows, columns=["StormID", "datetime", "MSLP_true", "MWS_true"]).sort_values(["StormID", "datetime"])

    if df_true_track.empty:
        df0 = df_pred_raw[df_pred_raw["lead_hour"] == 0].copy()
        if not df0.empty:
            p0 = df0.pivot_table(index=["StormID", "valid_time"], columns="Target", values="Pred", aggfunc="first").reset_index()
            p0 = p0.rename(columns={"valid_time": "datetime", "MSLP": "MSLP_true", "MWS": "MWS_true"})
            for c in ["MSLP_true", "MWS_true"]:
                if c not in p0.columns:
                    p0[c] = np.nan
            df_true_track = p0[["StormID", "datetime", "MSLP_true", "MWS_true"]].sort_values(["StormID", "datetime"])

    def _pred_target_long(target):
        d = df_pred_raw[df_pred_raw["Target"] == target].copy()
        return d[["StormID", "t0", "lead_hour", "valid_time", "Pred"]].rename(columns={"Pred": f"{target}_pred"})

    df_mslp_long = _pred_target_long("MSLP")
    df_mws_long  = _pred_target_long("MWS")

    df_pred_long = pd.merge(df_mslp_long, df_mws_long, on=["StormID", "t0", "lead_hour", "valid_time"], how="outer").sort_values(
        ["StormID", "t0", "lead_hour"]
    )

    lead_list = list(range(0, max_forecast_hours + 1, 6))

    def _pred_wide(target):
        d = df_pred_raw[df_pred_raw["Target"] == target].copy()
        wide = d.pivot_table(index=["StormID", "t0"], columns="lead_hour", values="Pred", aggfunc="first").reset_index()
        for h in lead_list:
            if h not in wide.columns:
                wide[h] = np.nan
        cols = ["StormID", "t0"] + lead_list
        wide = wide[cols].sort_values(["StormID", "t0"])
        wide = wide.rename(columns={h: f"+{h}h" for h in lead_list})
        return wide

    wide_mslp = _pred_wide("MSLP")
    wide_mws  = _pred_wide("MWS")

    engine = None
    try:
        import xlsxwriter  # noqa
        engine = "xlsxwriter"
    except Exception:
        try:
            import openpyxl  # noqa
            engine = "openpyxl"
        except Exception:
            engine = None

    excel_path_all = os.path.join(save_root, "rollout_all_storms.xlsx")

    if engine is None:
        base = os.path.splitext(excel_path_all)[0]
        df_true_track.to_csv(base + "_True_track_all.csv", index=False)
        df_pred_long.to_csv(base + "_Pred_long_all.csv", index=False)
        wide_mslp.to_csv(base + "_Pred_wide_MSLP.csv", index=False)
        wide_mws.to_csv(base + "_Pred_wide_MWS.csv", index=False)
        print(f"⚠️ Excel 엔진이 없어 CSV로 저장했습니다: {base}_*.csv")
    else:
        with pd.ExcelWriter(excel_path_all, engine=engine) as writer:
            df_true_track.to_excel(writer, sheet_name="True_track_all", index=False)
            df_pred_long.to_excel(writer, sheet_name="Pred_long_all", index=False)
            wide_mslp.to_excel(writer, sheet_name="Pred_wide_MSLP", index=False)
            wide_mws.to_excel(writer,  sheet_name="Pred_wide_MWS",  index=False)
        print(f"✅ Plot-friendly rollout Excel saved: {excel_path_all}")

    return excel_path_all

def plot_rollouts(pred_records, df_test, test_tag, T, save_root, MAX_FORECAST_HOURS, te_meta=None):
    plot_dir = os.path.join(save_root, f"plots_test_{test_tag}")
    os.makedirs(plot_dir, exist_ok=True)

    legend_path = os.path.join(save_root, "legend_rollout.png")
    if not os.path.exists(legend_path):
        save_legend(
            legend_path,
            labels=["True", "Predicted", "Missing pred"],
            colors=["black", "blue", "magenta"],
            markers=["o", "o", "x"]
        )

    required_leads = list(range(0, MAX_FORECAST_HOURS + 1, 6))   # 0,6,12,...,48
    required_set = set(required_leads)

    for (basin, tid), df_track in df_test.groupby(["basin", "id"]):
        tid_key = f"{basin}{str(tid).zfill(2)}"
        df_track = df_track.sort_values("datetime").copy()
        df_track["datetime"] = pd.to_datetime(df_track["datetime"])

        storm_start = pd.to_datetime(df_track["datetime"].min())
        storm_end   = pd.to_datetime(df_track["datetime"].max())

        rows = pred_records.get(tid_key, [])
        if rows:
            df_tid = pd.DataFrame(rows, columns=["t0", "lead_hour", "target", "y_true", "y_pred"])
            df_tid["t0"] = pd.to_datetime(df_tid["t0"])
            df_tid["lead_hour"] = df_tid["lead_hour"].astype(int)
        else:
            df_tid = pd.DataFrame(columns=["t0", "lead_hour", "target", "y_true", "y_pred"])

        for target_name in ["MWS", "MSLP"]:
            fig, ax = plt.subplots(figsize=(10, 6))

            if target_name == "MWS":
                ax.plot(df_track["datetime"], df_track["mws"], "k-", linewidth=2)
                ax.scatter(df_track["datetime"], df_track["mws"], c="black", s=25, marker="o")
                y_label = "MWS (kt)"
                true_vals = df_track["mws"].to_numpy(dtype=float)
                obs_col = "mws"
            else:
                ax.plot(df_track["datetime"], df_track["mslp"], "k-", linewidth=2)
                ax.scatter(df_track["datetime"], df_track["mslp"], c="black", s=25, marker="o")
                y_label = "MSLP (hPa)"
                true_vals = df_track["mslp"].to_numpy(dtype=float)
                obs_col = "mslp"

            pred_vals_all = []

            for dt in df_track["datetime"].to_numpy():
                dt = pd.Timestamp(dt)
                obs_val = float(df_track.loc[df_track["datetime"] == dt, obs_col].values[0])

                df_roll = df_tid[
                    (df_tid["t0"] == dt) & (df_tid["target"] == target_name)
                ].sort_values("lead_hour").copy()

                if df_roll.empty:
                    ax.scatter(dt, obs_val, marker="x", c="magenta", s=120, linewidths=2)
                    continue

                lead_set = set(df_roll["lead_hour"].astype(int).tolist())
                if not required_set.issubset(lead_set):
                    ax.scatter(dt, obs_val, marker="x", c="magenta", s=120, linewidths=2)
                    continue

                df_roll = df_roll[df_roll["lead_hour"].isin(required_leads)].copy()
                df_roll["lead_hour"] = pd.Categorical(df_roll["lead_hour"], categories=required_leads, ordered=True)
                df_roll = df_roll.sort_values("lead_hour")

                if np.any(~np.isfinite(df_roll["y_pred"].to_numpy(dtype=float))):
                    ax.scatter(dt, obs_val, marker="x", c="magenta", s=120, linewidths=2)
                    continue

                times = [dt + pd.Timedelta(hours=int(hh)) for hh in df_roll["lead_hour"].astype(int).tolist()]
                y_pred = df_roll["y_pred"].astype(float).tolist()

                clipped = [(t, y) for (t, y) in zip(times, y_pred) if t <= storm_end]
                if len(clipped) == 0:
                    ax.scatter(dt, obs_val, marker="x", c="magenta", s=120, linewidths=2)
                    continue

                times_c, y_pred_c = zip(*clipped)
                pred_vals_all.extend(list(y_pred_c))

                ax.plot(times_c, y_pred_c, "--", color="blue", alpha=0.3, linewidth=1)
                ax.scatter(times_c, y_pred_c, c="blue", s=12, marker="o", alpha=0.3)

            y_all = np.concatenate([true_vals, np.asarray(pred_vals_all, dtype=float)]) if len(pred_vals_all) > 0 else true_vals

            pad = 10.0
            y_min = np.floor((np.nanmin(y_all) - pad) / 10.0) * 10.0
            y_max = np.ceil((np.nanmax(y_all) + pad) / 10.0) * 10.0
            if target_name == "MWS":
                y_min = max(0.0, y_min)

            ax.yaxis.set_major_locator(mticker.MultipleLocator(10))
            ax.yaxis.set_minor_locator(mticker.MultipleLocator(5))
            ax.set_ylim(y_min, y_max)

            ax.set_xlabel("Datetime")
            ax.set_ylabel(y_label)
            ax.set_xlim(storm_start, storm_end)

            ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
            ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))
            ax.tick_params(axis="x", which="minor", labelbottom=False)

            ax.grid(True, which="major", linestyle="--", alpha=0.8)
            ax.grid(True, which="minor", linestyle=":", alpha=0.5)
            plt.setp(ax.get_xticklabels(), rotation=90, ha="center")

            fig.tight_layout()
            out_path = os.path.join(plot_dir, f"{tid_key}_{target_name}_rollout.png")
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)

    print(f"🖼️ All rollout plots saved in: {plot_dir}")
    gc.collect()

# ================== 11. Run All Folds (HP Grid) (REWRITE CLEAN) ==================
RUN_PI = False  # 요청대로 비활성 유지

def is_experiment_done(save_root):
    needed_files = ["model_physics_inspired_IR.keras", "rmse_overall.csv", "r2_overall.csv", "rollout_all_storms.xlsx"]
    for fname in needed_files:
        if not os.path.exists(os.path.join(save_root, fname)):
            return False
    return True

all_years = list(range(2016, 2025))
MAX_FORECAST_HOURS = 48
base_dir = "/content/drive/MyDrive/Colab/results_manual_irwvln"

def get_split(test_year):
    val_year = 2024 if test_year == 2016 else test_year - 1
    train_years = [y for y in all_years if y not in [test_year, val_year]]
    return train_years, [val_year], [test_year]

dec_filters_list = [48]
dropout_list     = [0.1, 0.2]
lr_list          = [1e-4, 2e-4]

for test_year in all_years:
    # split(train/val/test years) 바뀔 때만 전처리 캐시 초기화
    _PREPROC_CACHE.clear()

    cleanup_tf()
    K.clear_session()
    gc.collect()

    train_years, val_years, test_years = get_split(test_year)
    split_tag = f"train_{'-'.join(map(str, train_years))}_val_{val_years[0]}_test_{test_years[0]}"

    for dec_filters in dec_filters_list:
        for dropout_rate in dropout_list:
            for learning_rate in lr_list:
                # HP마다 모델/세션만 초기화 (전처리 캐시는 유지)
                reset_seeds(SEED + test_year)
                cleanup_tf()
                K.clear_session()
                gc.collect()

                lr_tag_map = {1e-4: "1e-4", 2e-4: "2e-4", 3e-4: "3e-4", 4e-4: "4e-4"}
                lr_tag = lr_tag_map.get(learning_rate, f"{learning_rate:.0e}")

                hp_tag = f"df{dec_filters}_do{int(dropout_rate * 100):02d}_lr{lr_tag}"
                save_root = os.path.join(base_dir, hp_tag, split_tag)
                os.makedirs(save_root, exist_ok=True)

                if is_experiment_done(save_root):
                    print(f"[SKIP] 이미 완료된 실험입니다. 결과 폴더: {save_root}")
                    continue

                print(
                    f"\n===== HP: dec_filters={dec_filters}, dropout={dropout_rate}, lr={learning_rate} | "
                    f"TRAIN={train_years}, VAL={val_years}, TEST={test_years} ====="
                )

                (
                    pred_records, df_test, test_tag, T,
                    env_key_order,
                    y_true_full_mws, y_true_full_mslp,
                    t0_te_mws, t0_te_mslp,
                    abs_scalers,
                    idx_te, te_meta, mask_te
                ) = run_train_val_test_multibranch(
                    train_years, val_years, test_years,
                    all_years, local_env_dir, MAX_FORECAST_HOURS, save_root,
                    dec_filters=dec_filters, dropout_rate=dropout_rate, learning_rate=learning_rate
                )

                if pred_records is None:
                    print(f"[WARN] pred_records가 None입니다. 이 조합은 산출물 생성 스킵: {save_root}")
                    cleanup_tf()
                    continue

                plot_rollouts(
                    pred_records, df_test, test_tag, T, save_root, MAX_FORECAST_HOURS,
                    te_meta=[te_meta[i] for i in idx_te]
                )

                save_rollouts_to_excel(
                    pred_records, save_root, df_test=df_test, max_forecast_hours=MAX_FORECAST_HOURS
                )

                if not RUN_PI:
                    print("[SKIP] RUN_PI=False → Permutation Importance(PI) 계산/저장 생략")

                cleanup_tf()
                gc.collect()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[SKIP] 2016.tar 이미 추출됨 → /content/env_data/background
[SKIP] 2017.tar 이미 추출됨 → /content/env_data/background
[SKIP] 2018.tar 이미 추출됨 → /content/env_data/background
[SKIP] 2019.tar 이미 추출됨 → /content/env_data/background
[SKIP] 2020.tar 이미 추출됨 → /content/env_data/background
[SKIP] 2021.tar 이미 추출됨 → /content/env_data/background
[SKIP] 2022.tar 이미 추출됨 → /content/env_data/background
[SKIP] 2023.tar 이미 추출됨 → /content/env_data/background
[SKIP] 2024.tar 이미 추출됨 → /content/env_data/background
[RESOLUTION] {'ENV_RES_DEG': 0.25, 'ENV_WIN_PX': 73, 'ENV_SPAN_DEG': 18.25, 'IR_NATIVE_DEG': 0.018, 'IR_CROP_PX': 1014, 'CENTER_X': 600, 'CENTER_Y': 600, 'TARGET_H': 73, 'TARGET_W': 73, 'IR_POOL_K': 1, 'RES_MODE': 'env0.25_ir0.018_span18.250'}
[SKIP] 이미 완료된 실험입니다. 결과 폴더: /content/drive/MyDrive/Colab/results_manual_irwvln/df48_do10_lr1e-4/train_2017-2018-2019-2020-2021-2022-2023_val_

train samples(cum-delta | state[T-6,T] + future ENV):   0%|          | 0/30 [00:00<?, ?it/s]

val samples(cum-delta | state[T-6,T] + future ENV):   0%|          | 0/33 [00:00<?, ?it/s]

test samples(cum-delta | state[T-6,T] + future ENV):   0%|          | 0/36 [00:00<?, ?it/s]

[Samples Created] Train=4805, Val=649, Test=988
[DEBUG] Selected target_id=10 cases=33 (all) | sort=t0_asc | out=/content/drive/MyDrive/Colab/results_manual_irwvln/df48_do10_lr1e-4/train_2016-2019-2020-2021-2022-2023-2024_val_2017_test_2018/debug
[DEBUG] Saved field-only plots -> /content/drive/MyDrive/Colab/results_manual_irwvln/df48_do10_lr1e-4/train_2016-2019-2020-2021-2022-2023-2024_val_2017_test_2018/debug/fields
[DEBUG] Saved standalone colorbars -> /content/drive/MyDrive/Colab/results_manual_irwvln/df48_do10_lr1e-4/train_2016-2019-2020-2021-2022-2023-2024_val_2017_test_2018/debug/colorbars
[DEBUG] Saved target_id=10 all t0 plots.
[PREPROC CACHE] Stored split-level preprocessed data
[AUX DIM] track_feat_dim=12
[ENV ORDER] ['ERH', 'EDIVE', 'EVORT', 'EVWS', 'ETHETA', 'SST', 'TOPO']
[ENV Tin] 10 = [-6, 0, +6, ..., +H]
[HP->WIDTHS] dec_filters=48 scale=1 | env_latent_dim=96 fused_channels=134 | MHA heads=8, keydim=32 (width=256) | MLP=384 | ConvLSTM df/df2=48/24
Epoch 1/100
100/100 -